# Reportmaker

In [1]:
# === WBE XOR Report (Jainil Shah) ===
"""
WBE (Whole Brain Emulation) XOR Network Comparison Report Generator

This script compares two neural network simulations (ground truth vs submission) that implement
XOR logic using spiking neurons. It analyzes multiple aspects of neural activity including:
- Spike patterns and timing
- Membrane potentials
- Inter-spike intervals
- Cross-correlations between neurons
- Behavioral performance (XOR logic accuracy)

The output is a comprehensive PDF report with visualizations and metrics.
"""

import io, math, warnings, json
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd

# =============================================================================
# Dependency setup (auto-install missing packages)
# =============================================================================
def _ensure_imports():
    """
    Automatically check for and install required packages if they're missing.
    This ensures the script can run even if some dependencies aren't installed.

    Returns:
        tuple: All imported modules needed for the analysis
    """
    import importlib, importlib.util, subprocess, sys as _sys

    # List of all packages required for the analysis
    required_pkgs = [
        "numpy",        # Numerical computing
        "pandas",       # Data manipulation
        "matplotlib",   # Plotting
        "seaborn",      # Statistical visualization
        "h5py",         # HDF5 file reading
        "scipy",        # Scientific computing (statistics, signal processing)
        "scikit-learn", # Machine learning utilities
        "reportlab"     # PDF generation
    ]

    # Check which packages are missing from the current Python environment
    missing = [p for p in required_pkgs if importlib.util.find_spec(p) is None]
    if missing:
        print(f"[setup] installing {' '.join(missing)} …")
        # Install missing packages using pip
        subprocess.check_call([_sys.executable, "-m", "pip", "install", *missing])
        print("[setup] install complete.\n")
    else:
        print("[setup] all requirements already satisfied.\n")

    # Import all modules after ensuring they're installed
    np = importlib.import_module("numpy")
    pd = importlib.import_module("pandas")
    plt = importlib.import_module("matplotlib.pyplot")
    sns = importlib.import_module("seaborn")
    h5py = importlib.import_module("h5py")

    # Import specific scipy submodules for statistics and signal processing
    scipy = importlib.import_module("scipy")
    scipy_stats = importlib.import_module("scipy.stats")
    scipy_signal = importlib.import_module("scipy.signal")

    # Import sklearn metrics for computing errors
    sklearn_metrics = importlib.import_module("sklearn.metrics")

    # Import ReportLab components for PDF generation
    reportlab = importlib.import_module("reportlab")
    from reportlab.lib.pagesizes import letter, A4
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image, Table, TableStyle, PageBreak
    from reportlab.lib.units import inch
    from reportlab.lib import colors
    from reportlab.lib.enums import TA_CENTER, TA_LEFT

    # Display version information for debugging
    print("[setup] package versions:")
    try:
        from importlib.metadata import version as _ver
    except:
        _ver = lambda x: "unknown"

    for name in required_pkgs:
        print(f"  - {name:<13} {_ver(name)}")
    print("")

    return (np, pd, plt, sns, h5py, scipy, scipy_stats, scipy_signal, sklearn_metrics,
            SimpleDocTemplate, Paragraph, Spacer, Image, Table, TableStyle, PageBreak,
            inch, letter, getSampleStyleSheet, colors, ParagraphStyle, TA_CENTER, TA_LEFT)

# Load all dependencies when the script starts
(np, pd, plt, sns, h5py, scipy, scipy_stats, scipy_signal, sklearn_metrics,
 SimpleDocTemplate, Paragraph, Spacer, Image, Table, TableStyle, PageBreak,
 inch, letter, getSampleStyleSheet, colors, ParagraphStyle, TA_CENTER, TA_LEFT) = _ensure_imports()

# Import specific statistical and signal processing functions we'll use frequently
from scipy.stats import ks_2samp, pearsonr, wasserstein_distance, f as f_dist
from scipy.signal import medfilt, find_peaks, lfilter, convolve
from sklearn.metrics import mean_squared_error
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D

# Suppress warnings that might clutter the output
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message=".*verbose is deprecated.*", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# =============================================================================
# Utility Functions
# =============================================================================
def _log(msg, verbose=True):
    """
    Print a status message if verbose mode is enabled.
    Used throughout the script to provide progress updates.

    Args:
        msg (str): Message to print
        verbose (bool): Whether to actually print the message
    """
    if verbose:
        print(msg)

def _fmt_decimal(x, decimals=3):
    """
    Format a number to a specified number of decimal places.
    Handles special cases like NaN, infinity, and very small values.

    Args:
        x: Number to format (can be None, NaN, or any numeric type)
        decimals (int): Number of decimal places to show

    Returns:
        Formatted number or NaN if input is invalid
    """
    # Check if the input is None or not a finite number
    if x is None or (isinstance(x, float) and not np.isfinite(x)):
        return np.nan

    try:
        val = float(x)
        # For very small values (less than 0.001), use more precision
        if abs(val) < 1e-3 and val != 0:
            # Convert to string with high precision
            str_val = f"{val:.10f}"
            # Find the first non-zero digit after the decimal point
            for i, c in enumerate(str_val.split('.')[1]):
                if c != '0':
                    # Show 3 significant digits after the leading zeros
                    return round(val, i + 3)
        # For normal values, use standard rounding
        return round(val, decimals)
    except:
        return x

def _fig_to_png_bytes(fig, dpi=150):
    """
    Convert a matplotlib figure to PNG bytes for embedding in PDF.

    Args:
        fig: Matplotlib figure object
        dpi (int): Resolution for the PNG image

    Returns:
        BytesIO: PNG image data as bytes
    """
    bio = io.BytesIO()
    fig.savefig(bio, format="png", dpi=dpi, bbox_inches="tight")
    plt.close(fig)  # Close the figure to free memory
    bio.seek(0)  # Reset to beginning of byte stream
    return bio

def _table(df, col_widths=None, style='normal', compact_cols=False):
    """
    Create a formatted ReportLab table from a pandas DataFrame.
    Automatically truncates long neuron names for better display.

    Args:
        df: Pandas DataFrame to convert
        col_widths: List of column widths (auto-calculated if None)
        style: Table style ('normal', 'compact', 'narrow')
        compact_cols: Whether to use compact column spacing

    Returns:
        Table: ReportLab Table object
    """
    df_formatted = df.copy()

    # Shorten long column/row names for better display
    for col in df_formatted.columns:
        # Format numeric columns to 3 decimal places
        if df_formatted[col].dtype in [np.float64, np.float32]:
            df_formatted[col] = df_formatted[col].apply(lambda x: _fmt_decimal(x, 3))
        # Remove redundant suffixes from neuron names
        elif df_formatted[col].dtype == object:
            df_formatted[col] = df_formatted[col].astype(str).str.replace('_spike_train', '')
            df_formatted[col] = df_formatted[col].astype(str).str.replace('_membrane_potential', '')

    # Rename columns to shorter versions if they're too long
    new_cols = []
    for col in df_formatted.columns:
        col_str = str(col)
        # Remove redundant suffixes
        col_str = col_str.replace('_spike_train', '').replace('_membrane_potential', '')
        # Shorten "Neuron" prefixes
        # if col_str.startswith('Neuron'):
        #     col_str = 'Neuron'
        new_cols.append(col_str)
    df_formatted.columns = new_cols

    # Convert DataFrame to list format for ReportLab
    data = [list(df_formatted.columns)] + df_formatted.astype(str).values.tolist()

    # Auto-calculate column widths if not provided
    if col_widths is None:
        if compact_cols:
            total_cols = len(df_formatted.columns)
            available_width = min(7.5 * inch, 1.0 * inch * total_cols)
        else:
            available_width = 7.0 * inch  # Standard width
        col_widths = [available_width / len(df_formatted.columns)] * len(df_formatted.columns)

    # Create the table
    t = Table(data, colWidths=col_widths, hAlign="LEFT")

    # Set font size based on style
    if style == 'compact':
        font_size = 8
    elif style == 'narrow':
        font_size = 8
    else:
        font_size = 9

    # Apply table styling
    t.setStyle(TableStyle([
        ("BACKGROUND",(0,0),(-1,0),colors.lightgrey),  # Header background
        ("FONTNAME",(0,0),(-1,0),"Helvetica-Bold"),    # Header font
        ("FONTSIZE",(0,0),(-1,-1),font_size),          # All cells font size
        ("GRID",(0,0),(-1,-1),0.3,colors.grey),        # Grid lines
        ("ALIGN",(0,0),(-1,-1),"LEFT"),                # Left align
        ("VALIGN",(0,0),(-1,-1),"MIDDLE"),             # Middle vertical align
    ]))
    return t

def _p(text, styles):
    """
    Shorthand to create a paragraph with normal styling.

    Args:
        text (str): Text content
        styles: ReportLab styles dictionary

    Returns:
        Paragraph: Formatted paragraph object
    """
    return Paragraph(text, styles["Normal"])

def _bullets(lines, styles, width=6.8):
    """
    Create a bulleted list from text lines with proper formatting.
    Handles section headers and long lines that need wrapping.

    Args:
        lines: List of text lines
        styles: ReportLab styles dictionary
        width: Width of the bullet list in inches

    Returns:
        Table or Paragraph: Formatted bullet list
    """
    if not lines:
        return Paragraph("No issues detected.", styles["Normal"])

    table_data = []
    for ln in lines:
        if ln.startswith("==="):
            # Section headers (bold, no bullet)
            table_data.append([Paragraph(f"<b>{ln.replace('===', '').strip()}</b>", styles["Normal"])])
        elif ln.strip() == "":
            # Empty lines for spacing
            table_data.append([Paragraph("", styles["Normal"])])
        else:
            # Regular bullet points
            if len(ln) > 100:
                # Split long lines for better readability
                words = ln.split()
                current = ""
                for word in words:
                    if len(current) + len(word) < 100:
                        current += " " + word if current else word
                    else:
                        table_data.append([Paragraph(f"• {current}", styles["Normal"])])
                        current = word
                if current:
                    table_data.append([Paragraph(f"• {current}", styles["Normal"])])
            else:
                table_data.append([Paragraph(f"• {ln}", styles["Normal"])])

    if not table_data:
        return Paragraph("No issues detected.", styles["Normal"])

    return Table(table_data, colWidths=[width*inch])

# =============================================================================
# Data Loading and Preprocessing
# =============================================================================
def load_trials_new(h5file):
    """
    Load neural data from HDF5 file containing multiple trials.
    Each trial contains time series data for multiple neurons.

    Args:
        h5file (str): Path to HDF5 file

    Returns:
        list: List of DataFrames, one per trial
    """
    trials = []
    with h5py.File(h5file, "r") as f:
        # Find all trial keys (e.g., "trial_0", "trial_1", etc.)
        keys = sorted([k for k in f.keys() if k.startswith("trial_")],
                     key=lambda x: int(x.split("_")[1]))

        for k in keys:
            # Load the data array for this trial
            data = f[k]["data"][()]
            df = pd.DataFrame(data)

            # Get column names from HDF5 attributes if available
            if "columns" in f[k]["data"].attrs:
                cols = list(f[k]["data"].attrs["columns"])
                # Handle byte strings from HDF5
                if isinstance(cols[0], bytes):
                    cols = [c.decode('utf-8') if isinstance(c, bytes) else c for c in cols]
                df.columns = cols

            trials.append(df)

    return trials

def _detect_pulse(vec, lo, hi, eps=1e-12):
    """
    Detect a pulse (continuous non-zero signal) within a time window.
    Used to find when stimuli A and B are active.

    Args:
        vec: Signal vector (typically stimulus input)
        lo: Start of search window
        hi: End of search window
        eps: Threshold for considering a value non-zero

    Returns:
        tuple: (start_index, extra_pulses_count) or (None, 0) if no pulse
    """
    v = np.asarray(vec)
    # Find where signal is above threshold
    on = np.isfinite(v) & (np.abs(v) > eps)

    # Create window mask
    win = np.zeros_like(on, dtype=bool)
    lo = max(0, int(lo))
    hi = min(len(on)-1, int(hi))
    if lo <= hi:
        win[lo:hi+1] = on[lo:hi+1]

    # Find continuous runs of activity
    if not win.any():
        return None, 0

    # Detect start and end of each run using difference
    diff = np.diff(np.concatenate(([False], win, [False])).astype(int))
    starts = np.where(diff == 1)[0]
    ends = np.where(diff == -1)[0] - 1

    if len(starts) == 0:
        return None, 0

    # Find the longest run (main pulse)
    runs = list(zip(starts, ends))
    lengths = [(e - s + 1) for s, e in runs]
    best_i = int(np.argmax(lengths))
    s, e = runs[best_i]
    extra = max(0, len(runs) - 1)  # Count of additional pulses

    return int(s), extra

def preprocess_trials(gt_trials, sub_trials, config):
    """
    Extract metadata and prepare data structures for analysis.
    Identifies stimulus patterns, response windows, and organizes data.

    Args:
        gt_trials: Ground truth trial data
        sub_trials: Submission trial data
        config: Configuration dictionary

    Returns:
        dict: Organized data structure with metadata and concatenated views
    """
    # Configuration parameters for stimulus and response detection
    stim_lo, stim_hi = config.get("stim_window", (5, 60))
    resp_lo = config.get("behavior_resp_lo", 3)
    resp_hi = config.get("behavior_resp_hi", 40)

    # Process ground truth trials to extract metadata
    gt_meta = []
    sub_meta = []

    for i, df in enumerate(gt_trials):
        # Extract stimulus channels A and B
        a_stim = df["A_stim"].values if "A_stim" in df else np.zeros(len(df))
        b_stim = df["B_stim"].values if "B_stim" in df else np.zeros(len(df))

        # Detect when stimuli occur
        a_time, _ = _detect_pulse(a_stim, stim_lo, stim_hi, config.get("behavior_stim_eps", 1e-12))
        b_time, _ = _detect_pulse(b_stim, stim_lo, stim_hi, config.get("behavior_stim_eps", 1e-12))

        # Create pattern code: "00", "01", "10", or "11" based on which inputs are active
        a_bit = 1 if a_time is not None else 0
        b_bit = 1 if b_time is not None else 0
        pattern = f"{a_bit}{b_bit}"

        # Define response window based on stimulus timing
        if a_time is not None or b_time is not None:
            # Response window starts after the earliest stimulus
            anchor = min([t for t in (a_time, b_time) if t is not None])
            resp_window = (max(0, anchor + resp_lo), min(99, anchor + resp_hi))
        else:
            resp_window = None

        gt_meta.append({
            "trial_id": i,
            "pattern": pattern,
            "a_time": a_time,
            "b_time": b_time,
            "response_window": resp_window
        })

    # Similar processing for submission trials
    for i, df in enumerate(sub_trials):
        a_stim = df["A_stim"].values if "A_stim" in df else np.zeros(len(df))
        b_stim = df["B_stim"].values if "B_stim" in df else np.zeros(len(df))

        a_time, _ = _detect_pulse(a_stim, stim_lo, stim_hi, config.get("behavior_stim_eps", 1e-12))
        b_time, _ = _detect_pulse(b_stim, stim_lo, stim_hi, config.get("behavior_stim_eps", 1e-12))

        a_bit = 1 if a_time is not None else 0
        b_bit = 1 if b_time is not None else 0
        pattern = f"{a_bit}{b_bit}"

        if a_time is not None or b_time is not None:
            anchor = min([t for t in (a_time, b_time) if t is not None])
            resp_window = (max(0, anchor + resp_lo), min(99, anchor + resp_hi))
        else:
            resp_window = None

        sub_meta.append({
            "trial_id": i,
            "pattern": pattern,
            "a_time": a_time,
            "b_time": b_time,
            "response_window": resp_window
        })

    # Identify spike train and membrane potential columns
    spike_cols = [c for c in gt_trials[0].columns if "spike_train" in c]
    vm_cols = [c for c in gt_trials[0].columns if "membrane_potential" in c]

    # Create concatenated views of all trials for easier analysis
    # Add trial_id and pattern info for tracking
    gt_concat = pd.concat([df.assign(trial_id=i, pattern=m["pattern"], t_in_trial=np.arange(len(df)))
                           for i, (df, m) in enumerate(zip(gt_trials, gt_meta))],
                          ignore_index=True)
    sub_concat = pd.concat([df.assign(trial_id=i, pattern=m["pattern"], t_in_trial=np.arange(len(df)))
                           for i, (df, m) in enumerate(zip(sub_trials, sub_meta))],
                          ignore_index=True)

    # Organize all processed data into a structured dictionary
    METRIC_VARS_NEW = {
        "gt": {
            "trials": gt_trials,
            "meta": gt_meta,
            "concat": gt_concat
        },
        "sub": {
            "trials": sub_trials,
            "meta": sub_meta,
            "concat": sub_concat
        },
        "spike_cols": spike_cols,
        "vm_cols": vm_cols,
        "fs_hz": 1000.0,  # Sampling frequency in Hz
        "behavior_params": {
            "RESP_WINDOW_LO_MS": resp_lo,
            "RESP_WINDOW_HI_MS": resp_hi
        },
        "outputs_map": {"E": "Neuron_E_spike_train"}  # Output neuron for XOR
    }

    return METRIC_VARS_NEW

# =============================================================================
# ALL METRIC IMPLEMENTATIONS (WITH ALL FIXES)
# =============================================================================

# --- RASTER METRIC (WITH BETTER ORGANIZED INSIGHTS) ---
def compute_raster_metrics(V, config):
    """
    Compute spike raster plots and Jaccard similarity metrics.

    The Jaccard index measures spike train similarity as:
    J = |intersection| / |union| of spike times

    Generates two raster plots:
    1. Overlay: GT and SUB spikes on same plot
    2. Differences: Only mismatched spikes

    Args:
        V: Preprocessed data structure
        config: Configuration parameters

    Returns:
        dict: Jaccard metrics, raster plots, and insights
    """
    # Extract data
    gt = V["gt"]["concat"].copy()
    sub = V["sub"]["concat"].copy()
    spike_cols = V["spike_cols"]
    TRIAL_LEN = 100  # Standard trial length in samples

    # Ensure same length for comparison
    n_rows = min(len(gt), len(sub))
    gt = gt.iloc[:n_rows].reset_index(drop=True)
    sub = sub.iloc[:n_rows].reset_index(drop=True)

    # Build spike time indices for each neuron
    gt_times = [np.where(gt[c].to_numpy(dtype=int) == 1)[0] for c in spike_cols]
    sub_times = [np.where(sub[c].to_numpy(dtype=int) == 1)[0] for c in spike_cols]
    # XOR to find mismatched spike times
    diff_times = [np.setxor1d(g, s) for g, s in zip(gt_times, sub_times)]

    # Helper function to add colored pattern bands to plots
    def add_pattern_bands(ax, df, trial_len=100, alpha=0.10):
        """Add colored background bands to indicate stimulus patterns"""
        # Color scheme for different input patterns
        colors_pat = {"00":"#a6cee3",  # No inputs (light blue)
                      "01":"#b2df8a",  # B only (light green)
                      "10":"#fdbf6f",  # A only (light orange)
                      "11":"#cab2d6"}  # Both inputs (light purple)

        last_tid = int(df["trial_id"].iloc[-1])
        for tid in range(last_tid + 1):
            block = df[df["trial_id"] == tid]
            if block.empty: continue
            start = int(block.index.min())
            pat = str(block["pattern"].iloc[0])
            # Add colored rectangle for this trial's pattern
            rect = Rectangle((start, -0.5), trial_len, len(spike_cols),
                           color=colors_pat.get(pat, "#dddddd"), alpha=alpha, lw=0)
            ax.add_patch(rect)

    # Figure 1: Overlay raster plot (GT vs SUB)
    fig1, ax = plt.subplots(figsize=(14, 8))
    add_pattern_bands(ax, gt, trial_len=TRIAL_LEN, alpha=0.10)
    # Plot GT spikes in black
    ax.eventplot(gt_times, orientation="horizontal", linelengths=0.8, linewidths=0.9, colors="black")
    # Plot SUB spikes in red (slightly shorter for visibility)
    ax.eventplot(sub_times, orientation="horizontal", linelengths=0.6, linewidths=0.9, colors="red")

    # Format neuron labels
    neuron_labels = [c.replace("_spike_train", "") for c in spike_cols]
    ax.set_yticks(np.arange(len(spike_cols)))
    ax.set_yticklabels(neuron_labels)
    ax.set_xlabel("Sample index (concatenated)")
    ax.set_ylabel("Neuron")
    ax.set_title("Raster — GT vs SUB (all patterns shaded)")

    # Add legend
    handles = [Line2D([0],[0], color="black", lw=2, label="GT"),
               Line2D([0],[0], color="red", lw=2, label="SUB")]
    ax.legend(handles=handles, loc="upper right")

    # Add trial boundaries
    total_trials = int(np.ceil(n_rows / TRIAL_LEN))
    for k in range(1, total_trials):
        ax.axvline(k * TRIAL_LEN, color="0.85", lw=0.5, zorder=0)

    plt.tight_layout()
    overlay_png = _fig_to_png_bytes(fig1, dpi=150)

    # Figure 2: Differences-only raster plot
    fig2, ax = plt.subplots(figsize=(14, 8))
    add_pattern_bands(ax, gt, trial_len=TRIAL_LEN, alpha=0.10)
    # Plot only the mismatched spikes
    ax.eventplot(diff_times, orientation="horizontal", linelengths=0.8, linewidths=0.9, colors="purple")
    ax.set_yticks(np.arange(len(spike_cols)))
    ax.set_yticklabels(neuron_labels)
    ax.set_xlabel("Sample index (concatenated)")
    ax.set_ylabel("Neuron")
    ax.set_title("Raster — Differences Only (GT ⊕ SUB)")

    # Add trial boundaries
    for k in range(1, total_trials):
        ax.axvline(k * TRIAL_LEN, color="0.85", lw=0.5, zorder=0)

    plt.tight_layout()
    diff_png = _fig_to_png_bytes(fig2, dpi=150)

    # Compute Jaccard similarity metrics
    def jaccard(a_idx, b_idx):
        """Calculate Jaccard index between two spike trains"""
        # Handle empty cases
        if len(a_idx) == 0 and len(b_idx) == 0:
            return 1.0  # Both empty = perfect match
        if len(a_idx) == 0 or len(b_idx) == 0:
            return 0.0  # One empty = no overlap

        # Jaccard = intersection / union
        inter = len(np.intersect1d(a_idx, b_idx))
        uni = len(np.union1d(a_idx, b_idx))
        return inter / uni if uni > 0 else 0.0

    # Calculate Jaccard for each neuron
    J = np.array([jaccard(g, s) for g, s in zip(gt_times, sub_times)], float)
    mismatch_counts = {name: int(len(d)) for name, d in zip(spike_cols, diff_times)}
    meanJ = float(np.nanmean(J)) if np.isfinite(J).any() else float("nan")

    # Calculate per-pattern Jaccard
    pats = ["00","01","10","11"]
    per_pat = {}
    pattern_jaccard_list = []

    for p in pats:
        # Filter data by pattern
        gtp = gt[gt["pattern"]==p]
        subp = sub[sub["pattern"]==p]
        n = min(len(gtp), len(subp))
        gtp = gtp.iloc[:n]; subp=subp.iloc[:n]

        # Get spike times for this pattern
        gts = [np.where(gtp[c].to_numpy(int)==1)[0] for c in spike_cols]
        sbs = [np.where(subp[c].to_numpy(int)==1)[0] for c in spike_cols]

        # Calculate Jaccard per neuron for this pattern
        Jp = np.array([jaccard(a,b) for a,b in zip(gts,sbs)], float)
        mean_pattern_j = float(np.nanmean(Jp))
        per_pat[p] = mean_pattern_j

        if np.isfinite(mean_pattern_j):
            pattern_jaccard_list.append(mean_pattern_j)

    # Mean Jaccard across all patterns
    mean_pattern_jaccard = float(np.mean(pattern_jaccard_list)) if pattern_jaccard_list else np.nan

    # Build organized insights for the report
    insights = []
    insights.append("=== Overall Metrics ===")
    insights.append(f"Mean Jaccard (all neurons): {_fmt_decimal(meanJ, 3)}")
    insights.append(f"Mean Jaccard (across patterns): {_fmt_decimal(mean_pattern_jaccard, 3)}")
    insights.append("")
    insights.append("=== Per-Neuron Details ===")
    for name, val in zip(spike_cols, J):
        neuron_short = name.replace("_spike_train", "")
        insights.append(f"{neuron_short}: Jaccard={_fmt_decimal(val, 3)}, Mismatches={mismatch_counts[name]}")
    insights.append("")
    insights.append("=== Per-Pattern Summary ===")
    for p in pats:
        insights.append(f"Pattern {p}: Mean Jaccard={_fmt_decimal(per_pat[p], 3)}")

    return {
        "mean_jaccard": _fmt_decimal(meanJ, 3),
        "mean_pattern_jaccard": _fmt_decimal(mean_pattern_jaccard, 3),
        "jaccard_per_neuron": [_fmt_decimal(x, 3) for x in J.tolist()],
        "overlay_png": overlay_png,
        "diff_png": diff_png,
        "neuron_labels": neuron_labels,
        "insights": insights,
        "per_pattern_jaccard": {k: _fmt_decimal(v, 3) for k, v in per_pat.items()}
    }

# --- PSTH METRIC (WITH FIXED TRANSPOSED TABLE) ---
def compute_psth_metrics(V, config):
    """
    Compute Peri-Stimulus Time Histograms (PSTH) and response-aligned plots.

    PSTH shows average firing rate over time, aligned to stimulus onset.
    Computes correlation, RMSE, and bias between GT and SUB.

    Args:
        V: Preprocessed data structure
        config: Configuration parameters

    Returns:
        dict: PSTH metrics, plots, and insights
    """
    # Extract data
    GT_trials = V["gt"]["trials"]
    SB_trials = V["sub"]["trials"]
    GT_meta = V["gt"]["meta"]
    SB_meta = V["sub"]["meta"]
    SPIKE_COLS = V["spike_cols"]
    fs_hz = float(V["fs_hz"])

    # Configuration parameters
    BIN_MS = config.get("bin_size", 5)  # Bin size for PSTH in milliseconds
    ALIGN_PRE_MS = config.get("psth_align_pre", 10)  # Time before stimulus
    ALIGN_POST_MS = config.get("psth_align_post", 80)  # Time after stimulus
    PATTERNS = ("00", "01", "10", "11")
    RESP_ALIGN_PATTERNS = ("01", "10", "11")  # Patterns with actual stimuli

    # Helper functions for PSTH computation
    def _safe_pearson(a, b):
        """Calculate Pearson correlation with NaN handling"""
        a = np.asarray(a, float); b = np.asarray(b, float)
        if a.size != b.size or a.size == 0:
            return np.nan
        va = np.nanvar(a); vb = np.nanvar(b)
        if va <= 0 or vb <= 0:
            return np.nan
        am = np.nanmean(a); bm = np.nanmean(b)
        num = np.nansum((a - am) * (b - bm))
        den = math.sqrt(np.nansum((a - am)**2) * np.nansum((b - bm)**2))
        return float(num / den) if den > 0 else np.nan

    def _rmse(a, b):
        """Calculate root mean squared error"""
        a = np.asarray(a, float); b = np.asarray(b, float)
        if a.size != b.size or a.size == 0:
            return np.nan
        d = a - b
        return float(np.sqrt(np.nanmean(d*d)))

    def _bin_counts(vec, bin_ms, fs):
        """Bin spike counts into time bins"""
        x = np.asarray(vec, int)
        bin_size = int(max(1, round((bin_ms/1000.0) * fs)))
        nb = len(x) // bin_size
        if nb == 0:
            return np.zeros(0, float)
        # Reshape into bins and sum spikes per bin
        xx = x[: nb*bin_size].reshape(nb, bin_size).sum(axis=1).astype(float)
        return xx

    def _trial_ids_by_pattern(meta, pattern):
        """Get trial indices for a specific pattern"""
        return [i for i, m in enumerate(meta) if m["pattern"] == pattern]

    def _psth_per_pattern(trials, patterns, col, bin_ms, fs, meta_ref, meta_other=None):
        """Compute PSTH for each pattern"""
        out = {}
        for p in patterns:
            # Find trials with this pattern
            idx_ref = _trial_ids_by_pattern(meta_ref, p)
            if meta_other is not None:
                # Use only common trials between GT and SUB
                idx_other = _trial_ids_by_pattern(meta_other, p)
                use = sorted(set(idx_ref).intersection(idx_other))
            else:
                use = idx_ref

            # Collect binned spike counts for all trials
            mats = []
            for i in use:
                if i >= len(trials):
                    continue
                v = trials[i][col].to_numpy(int)
                mats.append(_bin_counts(v, bin_ms, fs))

            # Average across trials
            if mats:
                L = min(map(len, mats))
                mats = [m[:L] for m in mats]
                out[p] = np.nanmean(np.stack(mats, axis=0), axis=0)
            else:
                out[p] = None
        return out

    def _resp_aligned(trials, meta, col, pre_ms, post_ms):
        """Compute response-aligned PSTH"""
        pre = int(pre_ms)
        post = int(post_ms)
        winL = pre + post + 1
        out = {}

        for p in RESP_ALIGN_PATTERNS:
            rows = []
            for i, m in enumerate(meta):
                if m["pattern"] != p:
                    continue
                if m["a_time"] is None and m["b_time"] is None:
                    continue

                # Align to earliest stimulus
                anchor = min([t for t in (m["a_time"], m["b_time"]) if t is not None])
                s = max(0, anchor - pre)
                e = min(99, anchor + post)

                # Extract window and pad if needed
                vec = trials[i][col].to_numpy(int)
                w = vec[s:e+1].astype(float)
                pad_pre = (anchor - pre) - s
                pad_post = (anchor + post) - e
                w = np.r_[ [np.nan]*pad_pre, w, [np.nan]*pad_post ]

                if w.size < winL:
                    w = np.r_[w, [np.nan]*(winL - w.size)]
                elif w.size > winL:
                    w = w[:winL]
                rows.append(w)

            # Average across trials
            if rows:
                mean_curve = np.nanmean(np.stack(rows, axis=0), axis=0)
                t_axis = np.arange(-pre, post+1, 1)
                out[p] = (t_axis, mean_curve)
            else:
                out[p] = None
        return out

    # Compute PSTHs for all neurons and patterns
    GT_psths = {c: _psth_per_pattern(GT_trials, PATTERNS, c, BIN_MS, fs_hz, GT_meta, SB_meta)
                for c in SPIKE_COLS}
    SB_psths = {c: _psth_per_pattern(SB_trials, PATTERNS, c, BIN_MS, fs_hz, GT_meta, SB_meta)
                for c in SPIKE_COLS}

    # Generate PSTH plots
    psth_plots = []

    # Per-pattern grid plots for each neuron
    for col in SPIKE_COLS:
        fig, axes = plt.subplots(1, 4, figsize=(16, 3.5), sharey=True)
        fig.suptitle(f"PSTH per pattern — {col} (bin={BIN_MS} ms)")

        for j, p in enumerate(PATTERNS):
            g = GT_psths[col][p]; s = SB_psths[col][p]
            ax = axes[j]

            # Plot GT and SUB PSTHs
            if g is not None: ax.plot(g, label="GT", linewidth=1.8)
            if s is not None: ax.plot(s, label="SUB", linewidth=1.8, linestyle="--")

            ax.set_title(f"pattern {p}")
            ax.set_xlabel("bin")
            if j == 0: ax.set_ylabel("spikes/bin")
            ax.grid(alpha=0.25)

        axes[-1].legend(loc="upper right")
        plt.tight_layout(rect=[0, 0, 1, 0.93])
        psth_plots.append(_fig_to_png_bytes(fig, dpi=150))

    # Response-aligned plots for output neurons (C, D, E)
    FOCUS_NEURONS = ["Neuron_C_spike_train", "Neuron_D_spike_train", "Neuron_E_spike_train"]
    for col in FOCUS_NEURONS:
        if col in SPIKE_COLS:
            # Compute response-aligned data
            GT_resp = _resp_aligned(GT_trials, GT_meta, col, ALIGN_PRE_MS, ALIGN_POST_MS)
            SB_resp = _resp_aligned(SB_trials, SB_meta, col, ALIGN_PRE_MS, ALIGN_POST_MS)

            fig, axes = plt.subplots(1, len(RESP_ALIGN_PATTERNS), figsize=(14, 3.5), sharey=True)
            fig.suptitle(f"Response-aligned PSTH — {col} (0 = earliest input)")

            for j, p in enumerate(RESP_ALIGN_PATTERNS):
                ax = axes[j]
                g = GT_resp[p]; s = SB_resp[p]

                # Plot aligned responses
                if g is not None:
                    tg, yg = g
                    ax.plot(tg, yg, label="GT", linewidth=1.8)
                if s is not None:
                    ts, ys = s
                    ax.plot(ts, ys, label="SUB", linewidth=1.8, linestyle="--")

                ax.set_title(f"pattern {p}")
                ax.set_xlabel("ms rel. to input")
                if j == 0: ax.set_ylabel("spikes/sample")
                ax.axvline(0, color="k", lw=0.8, ls=":")  # Mark stimulus onset
                ax.grid(alpha=0.25)

            axes[-1].legend(loc="upper right")
            plt.tight_layout(rect=[0, 0, 1, 0.93])
            psth_plots.append(_fig_to_png_bytes(fig, dpi=150))

    # Compute numeric metrics for each neuron
    rows = []
    perpat = {}

    for col in SPIKE_COLS:
        # Concatenate all patterns for overall metrics
        gt_vecs, sb_vecs = [], []
        for p in PATTERNS:
            g = GT_psths[col][p]; s = SB_psths[col][p]
            if g is None or s is None:
                continue
            L = min(len(g), len(s))
            gt_vecs.append(g[:L]); sb_vecs.append(s[:L])

        # Calculate overall metrics
        if gt_vecs and sb_vecs:
            G_all = np.concatenate(gt_vecs)
            S_all = np.concatenate(sb_vecs)
            r_all = _safe_pearson(G_all, S_all)
            rmse_all = _rmse(G_all, S_all)
            bias_all = float(np.nanmean(G_all - S_all))
        else:
            r_all = rmse_all = bias_all = np.nan

        # Per-pattern metrics
        pp = {}
        for p in PATTERNS:
            g = GT_psths[col][p]; s = SB_psths[col][p]
            if (g is None) or (s is None) or (len(g)==0) or (len(s)==0):
                pp[p] = {"r": np.nan, "rmse": np.nan, "bias": np.nan}
            else:
                L = min(len(g), len(s))
                gg, ss = g[:L], s[:L]
                pp[p] = {
                    "r": _safe_pearson(gg, ss),
                    "rmse": _rmse(gg, ss),
                    "bias": float(np.nanmean(gg - ss))
                }
        perpat[col] = pp

        # Build metrics row
        neuron_name = col.replace("_spike_train", "")
        rows.append({
            "Neuron": neuron_name,
            "Overall_r": _fmt_decimal(r_all, 3),
            "Overall_RMSE": _fmt_decimal(rmse_all, 3),
            "Overall_bias": _fmt_decimal(bias_all, 3),
            # Per-pattern correlations
            "r_00": _fmt_decimal(pp["00"]["r"], 3),
            "r_01": _fmt_decimal(pp["01"]["r"], 3),
            "r_10": _fmt_decimal(pp["10"]["r"], 3),
            "r_11": _fmt_decimal(pp["11"]["r"], 3),
            # Per-pattern RMSEs
            "RMSE_00": _fmt_decimal(pp["00"]["rmse"], 3),
            "RMSE_01": _fmt_decimal(pp["01"]["rmse"], 3),
            "RMSE_10": _fmt_decimal(pp["10"]["rmse"], 3),
            "RMSE_11": _fmt_decimal(pp["11"]["rmse"], 3),
            # Per-pattern biases
            "bias_00": _fmt_decimal(pp["00"]["bias"], 3),
            "bias_01": _fmt_decimal(pp["01"]["bias"], 3),
            "bias_10": _fmt_decimal(pp["10"]["bias"], 3),
            "bias_11": _fmt_decimal(pp["11"]["bias"], 3),
        })

    psth_metrics = pd.DataFrame(rows)

    # Create transposed table for better display
    psth_metrics_display = psth_metrics.copy()
    psth_metrics_display.set_index('Neuron', inplace=True)
    psth_metrics_transposed = psth_metrics_display.T
    psth_metrics_transposed.reset_index(inplace=True)
    psth_metrics_transposed.rename(columns={'index': 'Metric'}, inplace=True)

    # Build insights for report
    def f3(x):
        """Format number for display"""
        return "n/a" if not np.isfinite(x) else f"{_fmt_decimal(x, 3)}"

    bullets = []
    for _, row in psth_metrics.iterrows():
        n = row["Neuron"]
        r = row["Overall_r"]
        rm = row["Overall_RMSE"]
        b = row["Overall_bias"]

        # Grade the correlation quality
        grade = ("excellent" if (np.isfinite(r) and r >= 0.98)
                 else "good" if (np.isfinite(r) and r >= 0.90)
                 else "weak" if np.isfinite(r) else "n/a")

        bullets.append(
            f"{n}: shape {grade} (r={f3(r)}), "
            f"RMSE={f3(rm)} spikes/bin, "
            f"bias={f3(b)}"
        )

    return {
        "psth_metrics": psth_metrics,
        "psth_metrics_transposed": psth_metrics_transposed,
        "psth_plots": psth_plots,
        "per_pattern": perpat,
        "bullets": bullets
    }

# Continue with remaining metrics in next part...
# --- KS METRIC (WITH SHORT NAMES AND FULL INSIGHTS) ---
def compute_ks_metrics(V, config):
    """
    Compute Kolmogorov-Smirnov test statistics for spike timing distributions.

    The KS test compares cumulative distribution functions (CDFs) of spike times
    between GT and SUB to detect timing shifts or rate changes.

    Key outputs:
    - KS statistic: Maximum vertical distance between CDFs (0=identical, 1=no overlap)
    - p-value: Statistical significance of the difference
    - Wasserstein distance: Average displacement of spike times

    Args:
        V: Preprocessed data structure
        config: Configuration parameters

    Returns:
        dict: KS statistics, issue bullets, and flagged patterns
    """
    # Extract data
    GT_trials = V["gt"]["trials"]
    SB_trials = V["sub"]["trials"]
    GT_meta = V["gt"]["meta"]
    SB_meta = V["sub"]["meta"]
    SPIKE_COLS = V["spike_cols"]
    TRIAL_LEN = 100
    RESP_LO_MS = int(V["behavior_params"]["RESP_WINDOW_LO_MS"])
    RESP_HI_MS = int(V["behavior_params"]["RESP_WINDOW_HI_MS"])

    # Configuration for KS analysis
    KS_WINDOW_MODE = config.get("ks_window_mode", "full")  # "full", "aligned", or "guarded_full"
    RESP_ALIGN_PATTERNS = ("01","10","11")  # Patterns with actual stimuli
    ALL_PATTERNS = ("00","01","10","11")
    INCLUDE_00_IN_OVERALL = False  # Whether to include null pattern in overall stats
    USE_WASSERSTEIN = True  # Include Wasserstein distance metric

    # Statistical thresholds for flagging issues
    ALPHA = 0.05  # Significance level for p-values
    KS_TOL = 0.05  # KS statistic threshold for flagging
    COUNT_ABS_TOL = 5  # Absolute spike count difference tolerance
    COUNT_FRAC_TOL = 0.10  # Fractional spike count difference tolerance
    MIN_POOLED_SPIKES = 40  # Minimum spikes needed for reliable statistics

    # Helper functions for KS analysis
    def _trial_anchor(meta_row):
        """Get the earliest stimulus time in a trial"""
        a = meta_row["a_time"]; b = meta_row["b_time"]
        if (a is None) and (b is None): return None
        return min([t for t in (a, b) if t is not None])

    def _response_window(meta_row):
        """Get the response window for a trial"""
        return meta_row["response_window"]

    def _where_from_percentile(t_star, lo, hi):
        """Categorize where in the window the max divergence occurs"""
        if lo is None or hi is None or hi <= lo:
            return "n/a"
        frac = (t_star - lo) / (hi - lo)
        if frac < 1/3: return "early"
        if frac < 2/3: return "mid"
        return "late"

    def _window_bounds_for_mode(meta_row):
        """Determine analysis window based on mode"""
        if KS_WINDOW_MODE == "aligned":
            # Align to response window
            rw = _response_window(meta_row)
            if rw is None:
                return None, None, True, None, None
            lo, hi = rw
            return lo, hi, True, RESP_LO_MS, RESP_HI_MS
        elif KS_WINDOW_MODE == "guarded_full":
            # Use most of trial but avoid edges
            FULL_GUARD_LO = 5
            FULL_GUARD_HI = 95
            return FULL_GUARD_LO, FULL_GUARD_HI, False, FULL_GUARD_LO, FULL_GUARD_HI
        elif KS_WINDOW_MODE == "full":
            # Use entire trial
            return 0, TRIAL_LEN - 1, False, 0, TRIAL_LEN - 1
        else:
            raise ValueError(f"Unknown KS_WINDOW_MODE={KS_WINDOW_MODE}")

    def _pool_times(trials, meta, col, patterns):
        """Pool spike times across trials for specified patterns"""
        pooled = []
        where_lo, where_hi = None, None

        for i, m in enumerate(meta):
            p = m["pattern"]
            if p not in patterns:
                continue

            # Get window bounds for this trial
            lo, hi, align_to_anchor, wlo, whi = _window_bounds_for_mode(m)
            if lo is None or hi is None or hi < lo:
                continue
            if where_lo is None:
                where_lo, where_hi = wlo, whi

            # Extract spike times in window
            vec = trials[i][col].to_numpy(int)
            idx_rel = np.where(vec[lo:hi+1] == 1)[0]
            if idx_rel.size == 0:
                continue
            abs_idx = idx_rel + lo

            # Optionally align to stimulus onset
            if align_to_anchor:
                anc = _trial_anchor(m)
                if anc is None:
                    aligned = abs_idx.astype(float)
                else:
                    aligned = abs_idx.astype(float) - float(anc)
            else:
                aligned = abs_idx.astype(float)

            pooled.append(aligned)

        if not pooled:
            return np.array([], dtype=float), where_lo, where_hi
        return np.concatenate(pooled).astype(float), where_lo, where_hi

    def _ecdf_at_max_gap(x, y):
        """Find where empirical CDFs have maximum divergence"""
        x = np.asarray(x, float); y = np.asarray(y, float)

        # Handle empty cases
        if x.size == 0 and y.size == 0:
            return 0.0, 0.0
        if x.size == 0 or y.size == 0:
            base = x if y.size == 0 else y
            t_star = float(np.median(base))
            sign = 1.0 if y.size > x.size else -1.0
            return t_star, float(sign)

        # Compute ECDFs on combined grid
        xs = np.sort(x); ys = np.sort(y)
        grid = np.unique(np.r_[xs, ys])
        Fx = np.searchsorted(xs, grid, side="right") / xs.size
        Fy = np.searchsorted(ys, grid, side="right") / ys.size
        diff = Fy - Fx
        i = int(np.argmax(np.abs(diff)))
        return float(grid[i]), float(diff[i])

    # Compute pooled KS statistics overall
    overall_rows = []
    if KS_WINDOW_MODE == "aligned":
        overall_patterns = RESP_ALIGN_PATTERNS if not INCLUDE_00_IN_OVERALL else ALL_PATTERNS
    else:
        overall_patterns = ALL_PATTERNS

    for col in SPIKE_COLS:
        # Pool spike times across patterns
        gt_times, wlo, whi = _pool_times(GT_trials, GT_meta, col, overall_patterns)
        sb_times, _, _ = _pool_times(SB_trials, SB_meta, col, overall_patterns)

        # Count spikes
        GT_spikes = int(gt_times.size)
        SUB_spikes = int(sb_times.size)
        rate_ratio = float(SUB_spikes / GT_spikes) if GT_spikes > 0 else (np.inf if SUB_spikes > 0 else 1.0)

        # Compute KS statistics
        if GT_spikes == 0 and SUB_spikes == 0:
            ks_stat, p_val = 0.0, 1.0
            t_star, sign_val = 0.0, 0.0
            wdist = 0.0
        else:
            if GT_spikes == 0 or SUB_spikes == 0:
                ks_stat, p_val = (1.0, 0.0)
            else:
                ks = ks_2samp(gt_times, sb_times, alternative="two-sided", mode="auto")
                ks_stat, p_val = float(ks.statistic), float(ks.pvalue)
            t_star, sign_val = _ecdf_at_max_gap(gt_times, sb_times)

            # Wasserstein distance (earth mover's distance)
            wdist = float(wasserstein_distance(gt_times, sb_times)) if USE_WASSERSTEIN and gt_times.size > 0 and sb_times.size > 0 else np.nan

        # Categorize the results
        where = _where_from_percentile(t_star, wlo, whi)
        effect = ("tiny" if ks_stat < 0.05 else
                 "small" if ks_stat < 0.1 else
                 "medium" if ks_stat < 0.2 else "large")

        # Use short neuron name for display
        neuron_name = col.replace("_spike_train", "")

        overall_rows.append(dict(
            Neuron=neuron_name,
            KS_stat=_fmt_decimal(ks_stat, 3),
            p_value=_fmt_decimal(p_val, 3),
            effect=effect,
            GT_spikes=GT_spikes,
            SUB_spikes=SUB_spikes,
            rate_ratio=_fmt_decimal(rate_ratio, 3),
            wasserstein=_fmt_decimal(wdist, 3),
            t_max=_fmt_decimal(t_star, 3),
            where=where,
            sign=_fmt_decimal(sign_val, 3)
        ))

    ks_df = pd.DataFrame(overall_rows)

    # Compute per-pattern KS statistics
    pp_rows = []
    for col in SPIKE_COLS:
        for p in ALL_PATTERNS:
            patterns = (p,)
            gt_times, wlo, whi = _pool_times(GT_trials, GT_meta, col, patterns)
            sb_times, _, _ = _pool_times(SB_trials, SB_meta, col, patterns)

            GT_spikes = int(gt_times.size)
            SUB_spikes = int(sb_times.size)

            # Compute statistics for this pattern
            if GT_spikes == 0 and SUB_spikes == 0:
                ks_stat, p_val = 0.0, 1.0
                t_star, sign_val = 0.0, 0.0
                wdist = 0.0
            else:
                if GT_spikes == 0 or SUB_spikes == 0:
                    ks_stat, p_val = (1.0, 0.0)
                else:
                    ks = ks_2samp(gt_times, sb_times, alternative="two-sided", mode="auto")
                    ks_stat, p_val = float(ks.statistic), float(ks.pvalue)
                t_star, sign_val = _ecdf_at_max_gap(gt_times, sb_times)
                wdist = float(wasserstein_distance(gt_times, sb_times)) if USE_WASSERSTEIN and gt_times.size > 0 and sb_times.size > 0 else np.nan

            where = _where_from_percentile(t_star, wlo, whi)
            neuron_name = col.replace("_spike_train", "")

            pp_rows.append(dict(
                Neuron=neuron_name,
                pattern=p,
                KS_stat=_fmt_decimal(ks_stat, 3),
                p_value=_fmt_decimal(p_val, 3),
                GT_spikes=GT_spikes,
                SUB_spikes=SUB_spikes,
                wasserstein=_fmt_decimal(wdist, 3),
                where=where,
                sign=_fmt_decimal(sign_val, 3)
            ))

    ks_pp_df = pd.DataFrame(pp_rows)

    # Build issue bullets with coaching hints
    def _count_issue(gt, sub):
        """Check if spike count difference exceeds tolerance"""
        gt = float(gt); sub = float(sub)
        delta = sub - gt
        tol = max(COUNT_ABS_TOL, COUNT_FRAC_TOL * max(1.0, gt))
        return abs(delta) >= tol, delta, tol

    def _timing_issue(ks_stat, p, gt_spk, sub_spk):
        """Check if timing distribution significantly differs"""
        if (gt_spk < MIN_POOLED_SPIKES) or (sub_spk < MIN_POOLED_SPIKES):
            return False, "low-power"
        if not np.isfinite(ks_stat) or not np.isfinite(p):
            return False, "n/a"
        if (ks_stat >= KS_TOL) and (p < ALPHA):
            return True, "shift"
        return False, "match"

    def _dir_word(where_txt, sign_val):
        """Describe direction of timing shift"""
        base = "no directional skew" if abs(float(sign_val)) < 1e-12 else ("SUB earlier vs GT" if float(sign_val) > 0 else "SUB later vs GT")
        wt = where_txt if isinstance(where_txt, str) else "n/a"
        return f"{base} (peak divergence {wt})"

    # Generate issue bullets with specific coaching for each neuron
    issue_bullets = []
    for _, row in ks_df.iterrows():
        n = row["Neuron"]
        ks = float(row["KS_stat"]) if np.isfinite(row["KS_stat"]) else np.nan
        pv = float(row["p_value"]) if np.isfinite(row["p_value"]) else np.nan
        gt = float(row["GT_spikes"]); sub = float(row["SUB_spikes"])
        rr = float(row["rate_ratio"]) if np.isfinite(row["rate_ratio"]) else np.nan
        where = row["where"]; sgn = float(row["sign"]) if np.isfinite(row["sign"]) else 0.0
        wdist = float(row["wasserstein"]) if USE_WASSERSTEIN and np.isfinite(row["wasserstein"]) else np.nan

        timing_bad, timing_flag = _timing_issue(ks, pv, gt, sub)
        count_bad, delta, tol = _count_issue(gt, sub)

        if not timing_bad and not count_bad:
            continue

        # Build detailed issue description
        parts = [f"{n}:"]
        if timing_bad:
            if USE_WASSERSTEIN and np.isfinite(wdist):
                parts.append(f"timing shift (KS={_fmt_decimal(ks,3)}, p={_fmt_decimal(pv,3)}, W={_fmt_decimal(wdist,3)}, {_dir_word(where, sgn)})")
            else:
                parts.append(f"timing shift (KS={_fmt_decimal(ks,3)}, p={_fmt_decimal(pv,3)}, {_dir_word(where, sgn)})")
        if count_bad:
            parts.append(f"count {'surplus' if delta>0 else 'deficit'} Δ={_fmt_decimal(delta,0)} (SUB {sub:.0f} vs GT {gt:.0f}, tol≈±{_fmt_decimal(tol,0)}, rate≈{_fmt_decimal(rr,3)})")

        # Add neuron-specific coaching hints
        hint = None
        if "E" in n:  # Output neuron E (XOR logic)
            if count_bad and delta > 0:
                hint = "tighten E suppression (↑inhibition or ↓excitatory gain), verify C↔D balance on 11."
            elif timing_bad:
                hint = "nudge upstream latencies (delay excitation and/or advance inhibition)."
        elif "C" in n or "D" in n:  # Intermediate neurons C and D
            if count_bad: hint = "re-tune C/D drive to keep level aligned under active patterns."
            elif timing_bad: hint = "adjust synaptic delays (C/D) to align onset to GT."
        else:  # Input neurons A and B
            if count_bad: hint = "match tonic level (gain scaling)."
            elif timing_bad: hint = "align onset latency (synaptic delay / threshold)."

        if hint:
            parts.append(hint)

        issue_bullets.append(" ; ".join(parts))

    if not issue_bullets:
        issue_bullets = ["All neurons align within KS/count tolerances; no coaching needed."]

    # Flag specific pattern issues with severity ranking
    flagged_bullets = []

    def _severity_row(r):
        """Calculate severity score for prioritization"""
        ks  = float(r["KS_stat"]) if np.isfinite(r["KS_stat"]) else 0.0
        pv  = float(r["p_value"]) if np.isfinite(r["p_value"]) else 1.0
        gt  = float(r["GT_spikes"]); sub = float(r["SUB_spikes"])

        # Timing severity
        t_sev = ks if (ks >= KS_TOL and pv < ALPHA and gt >= MIN_POOLED_SPIKES and sub >= MIN_POOLED_SPIKES) else 0.0

        # Count severity
        ok, delta, tol = _count_issue(gt, sub)
        c_sev = abs(delta) / max(1e-9, tol) if ok else 0.0

        return max(t_sev, c_sev)

    # Add severity scores and find worst cases
    ks_pp_df["severity"] = ks_pp_df.apply(_severity_row, axis=1)
    pp_bad = ks_pp_df[ks_pp_df["severity"] > 0].copy()

    if not pp_bad.empty:
        # Get worst pattern for each neuron
        idx = pp_bad.groupby("Neuron")["severity"].idxmax()
        worst = pp_bad.loc[idx]
        worst = worst.sort_values(["severity","Neuron"], ascending=[False, True])

        for _, row in worst.iterrows():
            n = row["Neuron"]
            p = row["pattern"]
            ks = float(row["KS_stat"]) if np.isfinite(row["KS_stat"]) else np.nan
            pv = float(row["p_value"]) if np.isfinite(row["p_value"]) else np.nan
            wdist = float(row["wasserstein"]) if np.isfinite(row["wasserstein"]) else np.nan
            where = row["where"]

            parts = [f"{n} pattern {p}:"]
            parts.append(f"KS={_fmt_decimal(ks,3)}, p={_fmt_decimal(pv,3)}")
            if np.isfinite(wdist):
                parts.append(f"W={_fmt_decimal(wdist,3)}")
            parts.append(f"divergence {where}")
            parts.append(f"(GT={int(row['GT_spikes'])}, SUB={int(row['SUB_spikes'])})")

            flagged_bullets.append(" ".join(parts))
    else:
        flagged_bullets = ["No per-pattern KS issues detected."]

    return {
        "ks_df": ks_df,
        "ks_pp_df": ks_pp_df,
        "issue_bullets": issue_bullets,
        "flagged_bullets": flagged_bullets
    }

# --- ISI METRIC (WITH FULL INSIGHTS) ---
def compute_isi_metrics(V, config):
    """
    Compute Inter-Spike Interval (ISI) distributions and statistics.

    ISI analysis reveals firing regularity and temporal patterns.
    Key metrics:
    - CV (coefficient of variation): Regularity measure (0=regular, >1=irregular)
    - Wasserstein distance: Overall ISI distribution difference

    Args:
        V: Preprocessed data structure
        config: Configuration parameters

    Returns:
        dict: ISI histograms, CV comparisons, and insights
    """
    # Extract data
    GTc = V["gt"]["concat"].copy()
    SBc = V["sub"]["concat"].copy()
    SPIKE_COLS = V["spike_cols"]

    # Configuration
    KEEP_CROSS_TRIAL = config.get("isi_keep_cross_trial", True)  # Include ISIs across trial boundaries
    MAX_KEEP_MS = config.get("isi_max_keep_ms", 600)  # Maximum ISI to keep (clip outliers)
    HIST_BINS = config.get("isi_hist_bins", 48)  # Number of histogram bins
    CV_BAR_WIDTH = 0.35  # Width of bars in CV plot

    # Thresholds for flagging issues
    W_SMALL, W_MED, W_LARGE = 15, 35, 60  # Wasserstein distance thresholds
    CV_FLAG = 0.10  # CV difference threshold
    MEAN_PCT_FLAG = 0.10  # Mean ISI percent change threshold
    ARTIFACT_DROP = 0.50  # Fraction of W from cross-trial artifacts

    def _isi_concatenated(df, col, keep_cross_trial=True, clip_hi=None):
        """Extract ISI values from spike train"""
        idx = np.where(df[col].to_numpy(int) == 1)[0]
        if idx.size < 2:
            return np.array([], float)

        # Calculate intervals between consecutive spikes
        isis = np.diff(idx).astype(float)

        # Optionally remove ISIs that cross trial boundaries
        if not keep_cross_trial:
            tid = df["trial_id"].to_numpy(int)
            same = tid[idx[1:]] == tid[idx[:-1]]
            isis = isis[same]

        # Clip very long ISIs if requested
        if clip_hi is not None and isis.size:
            isis = isis[isis <= float(clip_hi)]

        return isis

    def _robust_bins(a, b, nbins=48):
        """Create histogram bins covering both distributions"""
        if a.size==0 and b.size==0:
            return np.linspace(0, 1, nbins+1)
        lo = (np.nanmin(a) if a.size else np.nanmin(b))
        hi = (np.nanmax(b) if b.size else np.nanmax(a))
        if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
            hi = lo + 1.0
        return np.linspace(lo, hi, nbins+1)

    def _cv(x):
        """Calculate coefficient of variation (std/mean)"""
        x = np.asarray(x, float)
        if x.size < 2: return np.nan
        m = float(np.nanmean(x))
        return np.nan if m <= 0 else float(np.nanstd(x)/m)

    def _neuron_hint(name, mean_shift, cv_shift):
        """Generate coaching hints based on ISI changes"""
        base = name.split("_")[0]

        if base == "C":
            if (mean_shift is not None and mean_shift < 0) or (cv_shift is not None and cv_shift < 0):
                return "C slightly overactive/steady → reduce A→C/B→C a bit or raise C threshold."
            return "C a bit lazier/irregular → small ↑A→C/B→C or lower C threshold."

        if base == "D":
            return "Keep OR drive consistent; big shifts can leak into E."

        if base == "E":
            if mean_shift is not None and mean_shift < 0:
                return "E faster/steadier → recheck D→E vs C→E to preserve XOR suppression."
            return "E slower/less regular → verify inhibition timing and D drive."

        if base in ("A","B"):
            return "Inputs off vs GT → double-check stim pacing/level."

        return None

    # Compute ISIs for all neurons
    isis_gt = {}
    isis_sb = {}
    isis_gt_local = {}  # Within-trial only
    isis_sb_local = {}  # Within-trial only

    for col in SPIKE_COLS:
        # Full ISIs (may include cross-trial)
        isis_gt[col] = _isi_concatenated(GTc, col, keep_cross_trial=KEEP_CROSS_TRIAL, clip_hi=MAX_KEEP_MS)
        isis_sb[col] = _isi_concatenated(SBc, col, keep_cross_trial=KEEP_CROSS_TRIAL, clip_hi=MAX_KEEP_MS)
        # Local ISIs (within-trial only)
        isis_gt_local[col] = _isi_concatenated(GTc, col, keep_cross_trial=False, clip_hi=None)
        isis_sb_local[col] = _isi_concatenated(SBc, col, keep_cross_trial=False, clip_hi=None)

    # Figure 1: ISI histograms for each neuron
    fig, axs = plt.subplots(len(SPIKE_COLS), 1, figsize=(12, 3.5*len(SPIKE_COLS)), sharex=False)
    if len(SPIKE_COLS) == 1: axs = [axs]

    for ax, col in zip(axs, SPIKE_COLS):
        g = isis_gt[col]; s = isis_sb[col]
        bins = _robust_bins(g, s, HIST_BINS)

        # Plot overlapping histograms
        if g.size: ax.hist(g, bins=bins, alpha=0.70, density=True, label="GT", color='tab:blue')
        if s.size: ax.hist(s, bins=bins, alpha=0.50, density=True, label="SUB", color='tab:orange')

        # Title with configuration info
        title = f"ISI (concatenated) — {col}"
        if KEEP_CROSS_TRIAL: title += " • cross-trial included"
        if MAX_KEEP_MS is not None: title += f" • clipped ≤{MAX_KEEP_MS} ms"

        ax.set_title(title, fontsize=10)
        ax.set_ylabel("Density")
        ax.grid(alpha=0.25)
        ax.legend(loc="upper right")

    axs[-1].set_xlabel("Inter-spike interval (samples ≈ ms)")
    plt.tight_layout()
    hist_png = _fig_to_png_bytes(fig, dpi=150)

    # Figure 2: ISI CV comparison bars
    gt_cv = [ _cv(isis_gt[col]) for col in SPIKE_COLS ]
    sb_cv = [ _cv(isis_sb[col]) for col in SPIKE_COLS ]
    labels = [c.replace("_spike_train","") for c in SPIKE_COLS]
    x = np.arange(len(SPIKE_COLS))

    fig = plt.figure(figsize=(12, 5))
    plt.bar(x - CV_BAR_WIDTH/2, gt_cv, CV_BAR_WIDTH, label="GT", color='tab:blue')
    plt.bar(x + CV_BAR_WIDTH/2, sb_cv, CV_BAR_WIDTH, label="SUB", color='tab:orange')
    plt.xticks(x, labels)
    plt.ylabel("ISI CV (concatenated)")
    plt.title("ISI CV per neuron")
    plt.legend()
    plt.grid(axis='y', alpha=0.25)
    plt.tight_layout()
    cv_png = _fig_to_png_bytes(fig, dpi=150)

    # Compute detailed statistics and generate insights
    def _fmt(x, nd=3):
        """Format number for display"""
        return "n/a" if (x is None or not np.isfinite(x)) else f"{x:.{nd}f}"

    rows = []
    for col in SPIKE_COLS:
        # Get all ISI data
        g_all = _isi_concatenated(GTc, col, True, None)
        s_all = _isi_concatenated(SBc, col, True, None)
        g_loc = isis_gt_local[col]
        s_loc = isis_sb_local[col]

        # Calculate mean ISI statistics
        mean_gt = np.nanmean(g_all) if g_all.size else np.nan
        mean_sb = np.nanmean(s_all) if s_all.size else np.nan
        d_mean = (mean_sb - mean_gt) if (np.isfinite(mean_sb) and np.isfinite(mean_gt)) else np.nan
        rel_mean = (d_mean / mean_gt) if (np.isfinite(d_mean) and np.isfinite(mean_gt) and mean_gt!=0) else np.nan

        # Calculate CV statistics
        cv_gt = _cv(g_all)
        cv_sb = _cv(s_all)
        d_cv = (cv_sb - cv_gt) if (np.isfinite(cv_sb) and np.isfinite(cv_gt)) else np.nan

        # Calculate Wasserstein distance
        if wasserstein_distance and g_all.size and s_all.size:
            W = float(wasserstein_distance(g_all, s_all))

            # Check how much is due to cross-trial artifacts
            if g_loc.size and s_loc.size:
                W_local = float(wasserstein_distance(g_loc, s_loc))
                artifact_share = 0.0 if W <= 0 else max(0.0, (W - W_local) / W)
            else:
                W_local = np.nan; artifact_share = np.nan
        else:
            W = np.nan; W_local = np.nan; artifact_share = np.nan

        neuron_name = col.replace("_spike_train", "")
        rows.append(dict(
            Neuron=neuron_name,
            nGT=int(g_all.size),
            nSUB=int(s_all.size),
            meanGT=_fmt_decimal(mean_gt, 3),
            meanSUB=_fmt_decimal(mean_sb, 3),
            dMean=_fmt_decimal(d_mean, 3),
            relMean=_fmt_decimal(rel_mean, 3),
            cvGT=_fmt_decimal(cv_gt, 3),
            cvSUB=_fmt_decimal(cv_sb, 3),
            dCV=_fmt_decimal(d_cv, 3),
            W=_fmt_decimal(W, 3),
            W_local=_fmt_decimal(W_local, 3),
            artifact_share=_fmt_decimal(artifact_share, 3)
        ))

    stats_df = pd.DataFrame(rows)

    # Generate insight bullets
    bullets = []
    for _, r in stats_df.iterrows():
        name = r["Neuron"]
        W = r["W"]; dcv = r["dCV"]; rmean = r["relMean"]; art = r["artifact_share"]
        n_ok = (r["nGT"] >= 20 and r["nSUB"] >= 20)  # Enough data for reliable statistics

        # Categorize Wasserstein distance severity
        if np.isfinite(W):
            if W >= W_LARGE: sev = "large"
            elif W >= W_MED: sev = "moderate"
            elif W >= W_SMALL: sev = "small"
            else: sev = "tiny"
        else:
            sev = "n/a"

        # Note if most difference is from cross-trial artifacts
        art_tag = " (mostly cross-trial)" if (np.isfinite(art) and art >= ARTIFACT_DROP) else ""

        # Build issue descriptions
        calls = []
        if n_ok and np.isfinite(W) and W >= W_MED:
            calls.append(f"W={_fmt_decimal(W,1)} ms {sev}{art_tag}")
        if n_ok and np.isfinite(dcv) and abs(dcv) >= CV_FLAG:
            calls.append(f"ΔCV={_fmt_decimal(dcv,2):+}")
        if n_ok and np.isfinite(rmean) and abs(rmean) >= MEAN_PCT_FLAG:
            calls.append(f"Δmean≈{_fmt_decimal(rmean*100,0):+.0f}%")

        if calls:
            # Add neuron-specific hint
            hint = _neuron_hint(name,
                               r["dMean"] if np.isfinite(r["dMean"]) else None,
                               r["dCV"] if np.isfinite(r["dCV"]) else None)
            bullets.append(f"{name}: " + "; ".join(calls) + (f" — {hint}" if hint else ""))

    if not bullets:
        bullets = ["ISI: GT and SUB closely match across all neurons (tiny W shift; |ΔCV| < 0.10)."]

    return {
        "hist_png": hist_png,
        "cv_png": cv_png,
        "gt_cv": [_fmt_decimal(x, 3) for x in gt_cv],
        "sub_cv": [_fmt_decimal(x, 3) for x in sb_cv],
        "insights": bullets,
        "stats_df": stats_df
    }

# Continue with remaining metrics in next part...
# --- FANO METRIC (CLEANED COLUMNS) ---
def compute_fano_metrics(V, config):
    """
    Compute Fano factors to measure spike count variability.

    The Fano factor (variance/mean) measures neuronal variability:
    - FF = 1: Poisson process (random)
    - FF < 1: More regular than Poisson
    - FF > 1: More variable than Poisson

    Calculated over response windows for each stimulus pattern.

    Args:
        V: Preprocessed data structure
        config: Configuration parameters

    Returns:
        dict: Fano factors per neuron, comparison plot, and insights
    """
    # Extract data
    GT_trials = V["gt"]["trials"]
    SB_trials = V["sub"]["trials"]
    GT_meta = V["gt"]["meta"]
    SB_meta = V["sub"]["meta"]
    SPIKE_COLS = V["spike_cols"]

    PATS = ("00", "01", "10", "11")  # All stimulus patterns
    TRIAL_LEN = len(GT_trials[0]) if GT_trials else 100
    DEFAULT_00_WINDOW = (max(0, 5), min(TRIAL_LEN - 1, 95))  # Default window for null pattern

    # Thresholds for flagging differences
    DELTA_FANO_FLAG = 0.15  # Fano factor difference threshold
    DELTA_MEAN_FLAG = 0.15  # Mean spike count difference threshold

    def _trial_ids(meta, p):
        """Get trial indices for a specific pattern"""
        return [i for i, m in enumerate(meta) if m.get("pattern") == p]

    def _counts_in_window(trials, meta, col, ids):
        """Count spikes in response window for each trial"""
        out = []
        for i in ids:
            pat = meta[i].get("pattern")
            rw = meta[i].get("response_window")

            # Determine counting window
            if (rw is None) and (pat == "00"):
                # Use default window for null pattern
                lo, hi = DEFAULT_00_WINDOW
            elif (rw is None) or (rw[1] < rw[0]):
                out.append(np.nan); continue
            else:
                lo, hi = int(rw[0]), int(rw[1])

            # Count spikes in window
            vec = trials[i][col].to_numpy(int)
            lo = max(0, int(lo)); hi = min(len(vec) - 1, int(hi))
            out.append(float(vec[lo:hi+1].sum()) if hi >= lo else np.nan)

        a = np.array(out, float)
        return a[np.isfinite(a)]  # Remove NaN values

    def _pooled_counts(trials, meta, col, patterns):
        """Pool spike counts across all specified patterns"""
        bags = []
        for p in patterns:
            # Get common trial IDs between GT and SUB
            ids = sorted(set(_trial_ids(GT_meta, p)).intersection(_trial_ids(SB_meta, p)))
            if ids:
                bags.append(_counts_in_window(trials, meta, col, ids))
        return np.concatenate(bags) if bags else np.array([], float)

    def _fano(arr):
        """Calculate Fano factor with robust statistics"""
        arr = np.asarray(arr, float)
        n = arr.size
        if n < 2:
            return np.nan, np.nan, np.nan, 0

        mu = float(np.mean(arr))  # Mean spike count
        var = float(np.var(arr, ddof=1))  # Sample variance (unbiased)
        ff = (var / mu) if mu > 0 else np.nan  # Fano factor

        return mu, var, ff, n

    # Calculate Fano factors for each neuron
    rows = []
    for col in SPIKE_COLS:
        # Pool counts across all patterns
        gt_c = _pooled_counts(GT_trials, GT_meta, col, PATS)
        sb_c = _pooled_counts(SB_trials, SB_meta, col, PATS)

        # Calculate Fano factors
        mu_g, var_g, ff_g, n_g = _fano(gt_c)
        mu_s, var_s, ff_s, n_s = _fano(sb_c)

        # Calculate differences for flagging issues
        d_ff = (ff_s - ff_g) if (np.isfinite(ff_s) and np.isfinite(ff_g)) else np.nan
        d_mean = (mu_s - mu_g) if (np.isfinite(mu_s) and np.isfinite(mu_g)) else np.nan

        rows.append(dict(
            neuron = col.replace("_spike_train", ""),
            ff_gt = _fmt_decimal(ff_g, 3),
            ff_sub = _fmt_decimal(ff_s, 3),
            mean_gt = _fmt_decimal(mu_g, 3),
            mean_sub = _fmt_decimal(mu_s, 3),
            trials = int(min(n_g, n_s))
        ))

    df = pd.DataFrame(rows)

    # Create comparison plot with value labels
    labels = df["neuron"].tolist()
    x = np.arange(len(labels)); w = 0.35  # Bar width

    fig, ax = plt.subplots(figsize=(12, 5))
    bars1 = ax.bar(x - w/2, df["ff_gt"].to_numpy(), width=w, label="GT", color='tab:blue')
    bars2 = ax.bar(x + w/2, df["ff_sub"].to_numpy(), width=w, label="SUB", color='tab:orange')

    ax.set_xticks(x); ax.set_xticklabels(labels)
    ax.set_ylabel("Fano factor (Var/Mean)")
    ax.set_title("Fano Factor per Neuron — pooled over 00/01/10/11 (counts in response window)")
    ax.legend(loc="upper right")
    ax.grid(axis="y", alpha=0.2)

    def _annotate(ax, bars, vals):
        """Add value labels on top of bars"""
        for b, v in zip(bars, vals):
            txt = "n/a" if not np.isfinite(v) else f"{_fmt_decimal(v, 3)}"
            y = 0.0 if not np.isfinite(v) else float(v)
            ax.text(b.get_x() + b.get_width()/2, y + 0.02, txt,
                   ha="center", va="bottom", fontsize=8)

    _annotate(ax, bars1, df["ff_gt"].to_numpy())
    _annotate(ax, bars2, df["ff_sub"].to_numpy())

    plt.tight_layout()
    fano_png = _fig_to_png_bytes(fig, dpi=150)

    # Generate issue notes for significant differences
    notes = []
    for i, r in df.iterrows():
        gt_ff = r["ff_gt"]; sub_ff = r["ff_sub"]
        gt_mean = r["mean_gt"]; sub_mean = r["mean_sub"]

        # Calculate differences
        d_ff = (sub_ff - gt_ff) if (np.isfinite(sub_ff) and np.isfinite(gt_ff)) else np.nan
        d_mean = (sub_mean - gt_mean) if (np.isfinite(sub_mean) and np.isfinite(gt_mean)) else np.nan

        # Flag significant differences
        if (np.isfinite(d_ff) and abs(d_ff) > DELTA_FANO_FLAG) or \
           (np.isfinite(d_mean) and abs(d_mean) > DELTA_MEAN_FLAG):
            tag = []
            if np.isfinite(d_ff) and abs(d_ff) > DELTA_FANO_FLAG:
                tag.append(f"ΔFano={_fmt_decimal(d_ff, 2):+}")
            if np.isfinite(d_mean) and abs(d_mean) > DELTA_MEAN_FLAG:
                tag.append(f"ΔMean={_fmt_decimal(d_mean, 2):+}")
            notes.append(f"{r['neuron']}: " + ", ".join(tag))

    if not notes:
        notes = ["Fano factors align well across all neurons."]

    return {
        "fano_df": df,
        "fano_png": fano_png,
        "gt_fano": df["ff_gt"].values.tolist(),
        "sub_fano": df["ff_sub"].values.tolist(),
        "insights": notes
    }

# --- VM METRIC (UNCHANGED - WORKS WELL) ---
def compute_vm_metrics(V, config):
    """
    Generate comprehensive membrane potential (VM) visualizations.

    Creates multiple plot types for each neuron and pattern:
    - Stitched traces: Concatenated trials
    - Median traces: Average response per pattern
    - IQR bands: Inter-quartile range showing variability

    Args:
        V: Preprocessed data structure
        config: Configuration parameters

    Returns:
        dict: Collection of VM plots for all neurons and patterns
    """
    # Extract data
    GT_trials = V["gt"]["trials"]
    SB_trials = V["sub"]["trials"]
    GT_meta = V["gt"]["meta"]
    SB_meta = V["sub"]["meta"]
    VM_COLS = V["vm_cols"]  # Membrane potential columns
    TRIAL_LEN = len(GT_trials[0]) if GT_trials else 100
    PATTERNS = ("00","01","10","11")

    MAX_TRIALS_PER_PATTERN_STITCH = None  # Limit for stitching (None = all)

    def _trial_ids(meta, patt):
        """Get trial indices for a specific pattern"""
        return [i for i, m in enumerate(meta) if m.get("pattern") == patt]

    def _common_trial_ids(patt):
        """Get trial indices present in both GT and SUB"""
        g = set(_trial_ids(GT_meta, patt))
        s = set(_trial_ids(SB_meta, patt))
        return sorted(g.intersection(s))

    def _median_iqr_over_trials(trials, idxs, col):
        """Calculate median and inter-quartile range across trials"""
        if not idxs:
            return None, None, None

        mat = []
        for i in idxs:
            if i >= len(trials):
                continue
            vec = trials[i][col].to_numpy(float)
            if len(vec) >= TRIAL_LEN:
                mat.append(vec[:TRIAL_LEN])

        if not mat:
            return None, None, None

        # Stack trials and compute statistics
        M = np.vstack(mat)
        med = np.nanmedian(M, axis=0)
        q25 = np.nanpercentile(M, 25, axis=0)  # 25th percentile
        q75 = np.nanpercentile(M, 75, axis=0)  # 75th percentile

        return med, q25, q75

    def _stitch_series(trials, idxs, col, max_trials=None):
        """Concatenate multiple trials into one continuous series"""
        if not idxs:
            return np.array([]), np.array([])

        # Limit number of trials if specified
        use = idxs if (max_trials is None) else idxs[:max_trials]

        chunks = []
        for i in use:
            vec = trials[i][col].to_numpy(float)[:TRIAL_LEN]
            chunks.append(vec)

        if not chunks:
            return np.array([]), np.array([])

        y = np.concatenate(chunks)  # Stitched data
        t = np.arange(y.size, dtype=float)  # Time axis

        return y, t

    def _neuron_label(col):
        """Extract neuron name from column"""
        return col.replace("_membrane_potential", "")

    # Store all generated plots
    vm_plots = []

    # Generate ALL plots for each pattern and neuron combination
    for patt in PATTERNS:
        common_ids = _common_trial_ids(patt)

        for col in VM_COLS:
            neuron = _neuron_label(col)

            # Precompute median and IQR statistics
            med_gt, q25_gt, q75_gt = _median_iqr_over_trials(GT_trials, common_ids, col)
            med_sub, q25_sub, q75_sub = _median_iqr_over_trials(SB_trials, common_ids, col)

            # Plot 1: GT stitched trace + GT median
            y_gt, t_gt = _stitch_series(GT_trials, common_ids, col, MAX_TRIALS_PER_PATTERN_STITCH)
            fig, ax = plt.subplots(figsize=(12, 3.2))

            if y_gt.size:
                ax.plot(t_gt, y_gt, lw=0.6, label="GT stitched", alpha=0.9)

            if med_gt is not None:
                # Tile median to match stitched length
                reps = int(np.ceil(max(1, y_gt.size) / TRIAL_LEN))
                med_tile = np.tile(med_gt, reps)[: max(1, y_gt.size)]
                ax.plot(np.arange(med_tile.size), med_tile, lw=1.6, linestyle="--",
                       label="GT median", alpha=0.9)

            ax.set_title(f"{neuron} — GT stitched (pattern {patt}) + GT median")
            ax.set_xlabel("sample (ms)")
            ax.set_ylabel("V_m")
            ax.grid(alpha=0.25)
            ax.legend(loc="upper right")
            plt.tight_layout()
            vm_plots.append(_fig_to_png_bytes(fig, dpi=150))

            # Plot 2: SUB stitched trace + SUB median
            y_sub, t_sub = _stitch_series(SB_trials, common_ids, col, MAX_TRIALS_PER_PATTERN_STITCH)
            fig, ax = plt.subplots(figsize=(12, 3.2))

            if y_sub.size:
                ax.plot(t_sub, y_sub, lw=0.6, color="tab:orange", label="SUB stitched", alpha=0.9)

            if med_sub is not None:
                reps = int(np.ceil(max(1, y_sub.size) / TRIAL_LEN))
                med_tile = np.tile(med_sub, reps)[: max(1, y_sub.size)]
                ax.plot(np.arange(med_tile.size), med_tile, lw=1.6, linestyle="--",
                       color="tab:blue", label="SUB median", alpha=0.9)

            ax.set_title(f"{neuron} — SUB stitched (pattern {patt}) + SUB median")
            ax.set_xlabel("sample (ms)")
            ax.set_ylabel("V_m")
            ax.grid(alpha=0.25)
            ax.legend(loc="upper right")
            plt.tight_layout()
            vm_plots.append(_fig_to_png_bytes(fig, dpi=150))

            # Plot 3a: GT Median ± IQR
            x = np.arange(TRIAL_LEN)
            fig, ax = plt.subplots(figsize=(10, 3.0))

            if med_gt is not None:
                ax.plot(x, med_gt, lw=2.0, label="GT median")
                if (q25_gt is not None) and (q75_gt is not None):
                    ax.fill_between(x, q25_gt, q75_gt, alpha=0.25, label="GT IQR")

            ax.set_title(f"{neuron} — Median ± IQR (GT) — pattern {patt}")
            ax.set_xlabel("time within trial (ms)")
            ax.set_ylabel("V_m")
            ax.grid(alpha=0.25)
            ax.legend(loc="upper right")
            plt.tight_layout()
            vm_plots.append(_fig_to_png_bytes(fig, dpi=150))

            # Plot 3b: SUB Median ± IQR
            fig, ax = plt.subplots(figsize=(10, 3.0))

            if med_sub is not None:
                ax.plot(x, med_sub, lw=2.0, color="tab:orange", label="SUB median")
                if (q25_sub is not None) and (q75_sub is not None):
                    ax.fill_between(x, q25_sub, q75_sub, alpha=0.25, color="tab:orange", label="SUB IQR")

            ax.set_title(f"{neuron} — Median ± IQR (SUB) — pattern {patt}")
            ax.set_xlabel("time within trial (ms)")
            ax.set_ylabel("V_m")
            ax.grid(alpha=0.25)
            ax.legend(loc="upper right")
            plt.tight_layout()
            vm_plots.append(_fig_to_png_bytes(fig, dpi=150))

    return {"vm_plots": vm_plots}

# --- VM ZOOM/MISMATCH METRIC (IMPROVED) ---
def compute_vm_zoom_metrics(V, config):
    """
    Identify and visualize worst VM mismatches between GT and SUB.

    Finds trials with largest membrane potential differences and creates
    detailed comparison plots highlighting the discrepancies.

    Args:
        V: Preprocessed data structure
        config: Configuration parameters

    Returns:
        dict: Worst mismatch tables and detailed comparison plots
    """
    # Extract data
    GT_TRIALS = V["gt"]["trials"]
    SB_TRIALS = V["sub"]["trials"]
    GT_META = V["gt"]["meta"]
    SB_META = V["sub"]["meta"]
    VM_COLS = list(V["vm_cols"])
    PATTERNS = ("00", "01", "10", "11")
    FS_HZ = float(V.get("fs_hz", 1000.0))
    MS_PER_SAMPLE = 1000.0 / FS_HZ

    def _common_ids_for_pattern(patt):
        """Get trial indices present in both GT and SUB for a pattern"""
        gt_ids = {i for i, m in enumerate(GT_META) if str(m.get("pattern")) == patt}
        sb_ids = {i for i, m in enumerate(SB_META) if str(m.get("pattern")) == patt}
        return sorted(gt_ids.intersection(sb_ids))

    def _trial_full_stats(gt_df, sb_df, vm_cols):
        """Calculate mismatch statistics for a trial"""
        n = min(len(gt_df), len(sb_df))
        if n == 0:
            return {"neuron_col": None, "rms": 0.0, "peak": 0.0}

        best_col, best_rms, best_peak = None, 0.0, 0.0

        # Find neuron with worst mismatch
        for col in vm_cols:
            if (col not in gt_df.columns) or (col not in sb_df.columns):
                continue

            g = gt_df[col].to_numpy(dtype=float)[:n]
            s = sb_df[col].to_numpy(dtype=float)[:n]
            d = g - s  # Difference signal

            if d.size == 0:
                continue

            # Calculate error metrics
            rms = float(np.sqrt(np.nanmean(d * d)))  # Root mean square error
            peak = float(np.nanmax(np.abs(d)))  # Peak absolute error

            # Track worst neuron
            if rms > best_rms:
                best_rms = rms
                best_peak = peak
                best_col = col

        return {"neuron_col": best_col, "rms": best_rms, "peak": best_peak}

    def _pattern_of_trial(idx):
        """Get pattern for a trial index"""
        p = GT_META[idx].get("pattern", None)
        if p is None:
            p = SB_META[idx].get("pattern", None)
        return str(p) if p is not None else None

    # Table 1: Worst trial per pattern
    p_rows = []
    for patt in PATTERNS:
        ids = _common_ids_for_pattern(patt)
        if not ids:
            continue

        # Score all trials for this pattern
        scored = []
        for i in ids:
            s = _trial_full_stats(GT_TRIALS[i], SB_TRIALS[i], VM_COLS)
            if s["neuron_col"] is None:
                continue
            scored.append((i, s))

        if not scored:
            continue

        # Find worst trial (highest RMS error)
        trial_id, stat = max(scored, key=lambda x: x[1]["rms"])

        # Shorten neuron name for display
        neuron_short = stat["neuron_col"].replace("_membrane_potential", "")

        p_rows.append({
            "pattern": patt,
            "trial_id": int(trial_id),
            "Neuron": neuron_short,
            "RMS Δ": _fmt_decimal(stat["rms"], 3),
            "Peak |Δ|": _fmt_decimal(stat["peak"], 3)
        })

    pattern_worst_table = pd.DataFrame(p_rows, columns=[
        "pattern","trial_id","Neuron","RMS Δ","Peak |Δ|"
    ])

    # Table 2: Worst trial per neuron
    N = len(GT_TRIALS)
    n_rows = []

    for col in VM_COLS:
        best = {"trial": None, "patt": None, "rms": 0.0, "peak": 0.0}

        # Check all trials for this neuron
        for i in range(N):
            if i >= len(SB_TRIALS):
                continue

            gt_df, sb_df = GT_TRIALS[i], SB_TRIALS[i]
            if (col not in gt_df.columns) or (col not in sb_df.columns):
                continue

            stat = _trial_full_stats(gt_df, sb_df, [col])
            if stat["neuron_col"] is None:
                continue

            # Track worst trial for this neuron
            if stat["rms"] > best["rms"]:
                best.update({
                    "trial": i,
                    "patt": _pattern_of_trial(i),
                    "rms": float(stat["rms"]),
                    "peak": float(stat["peak"])
                })

        # Shorten neuron name for display
        neuron_short = col.replace("_membrane_potential", "")

        if best["trial"] is None:
            n_rows.append({
                "Neuron": neuron_short,
                "RMS Δ": _fmt_decimal(0.0, 3),
                "Peak |Δ|": _fmt_decimal(0.0, 3),
                "pattern": None,
                "trial_id": np.nan
            })
        else:
            n_rows.append({
                "Neuron": neuron_short,
                "RMS Δ": _fmt_decimal(best["rms"], 3),
                "Peak |Δ|": _fmt_decimal(best["peak"], 3),
                "pattern": best["patt"],
                "trial_id": int(best["trial"])
            })

    neuron_best_table = pd.DataFrame(n_rows, columns=[
        "Neuron","RMS Δ","Peak |Δ|","pattern","trial_id"
    ])

    # Generate detailed plots for worst trials
    vm_zoom_plots = []

    # Collect unique (pattern, trial_id) pairs from tables
    to_plot = set()
    for df in (pattern_worst_table, neuron_best_table):
        if not df.empty:
            for _, r in df.iterrows():
                patt = r.get("pattern", None)
                tid = r.get("trial_id", None)
                if pd.notna(patt) and pd.notna(tid):
                    to_plot.add((str(patt), int(tid)))

    # Plot each selected trial with all neurons
    for patt, tid in sorted(to_plot):
        ids = _common_ids_for_pattern(patt)
        if tid not in ids:
            continue

        gt_df = GT_TRIALS[tid]
        sb_df = SB_TRIALS[tid]
        n = min(len(gt_df), len(sb_df))

        if n == 0:
            continue

        # Get time axis
        if "time" in gt_df.columns:
            tt = gt_df["time"].to_numpy(dtype=float)[:n]
            xlab = "Time (ms)"
        else:
            tt = np.arange(n, dtype=float) * MS_PER_SAMPLE
            xlab = "Time (ms, inferred)"

        # Create stacked subplot for all VM channels
        rows = len(VM_COLS)
        fig_h = max(3.5 * rows, 5.0)  # Taller plots for better visibility
        fig, axs = plt.subplots(rows, 1, figsize=(10, fig_h), sharex=True)
        if rows == 1:
            axs = [axs]

        for ax, c in zip(axs, VM_COLS):
            if (c not in gt_df.columns) or (c not in sb_df.columns):
                ax.axis("off")
                continue

            # Plot GT and SUB traces
            g = gt_df[c].to_numpy(dtype=float)[:n]
            s = sb_df[c].to_numpy(dtype=float)[:n]
            ax.plot(tt, g, lw=1.6, label="GT", color='tab:blue')
            ax.plot(tt, s, lw=1.6, ls="--", label="SUB", color='tab:orange')

            # Fill area between to highlight differences
            ax.fill_between(tt, g, s, alpha=0.12, color='purple')

            ax.set_ylabel("mV", fontsize=9)
            ax.set_title(c.replace("_membrane_potential", ""), fontsize=10)
            ax.grid(alpha=0.25)

        axs[-1].set_xlabel(xlab)

        # Add summary statistics to title
        stat = _trial_full_stats(gt_df, sb_df, VM_COLS)
        neuron_short_stat = stat['neuron_col'].replace("_membrane_potential", "") if stat['neuron_col'] else "n/a"
        note = f"worst={neuron_short_stat} • peak|Δ|={_fmt_decimal(stat['peak'],3)} • rmsΔ={_fmt_decimal(stat['rms'],3)}"

        fig.suptitle(
            f"Pattern {patt} — Trial {tid} • {note}",
            fontsize=11
        )
        axs[0].legend(loc="upper right", fontsize=9)
        plt.tight_layout(rect=[0, 0, 1, 0.94])
        vm_zoom_plots.append(_fig_to_png_bytes(fig, dpi=150))

    return {
        "pattern_worst_table": pattern_worst_table,
        "neuron_best_table": neuron_best_table,
        "vm_zoom_plots": vm_zoom_plots
    }

# Continue with remaining metrics in next part...
# --- PSP METRIC (Post-Synaptic Potential Detection) ---
def compute_psp_metrics(V, config):
    """
    Detect and count post-synaptic potentials (EPSPs and IPSPs) in VM traces.

    PSPs are small voltage deflections caused by synaptic inputs:
    - EPSP: Excitatory (positive deflection)
    - IPSP: Inhibitory (negative deflection)

    Uses peak detection after baseline subtraction to identify PSPs.

    Args:
        V: Preprocessed data structure
        config: Configuration parameters

    Returns:
        dict: PSP counts per pattern and neuron
    """
    # Extract data
    GT_trials = V["gt"]["trials"]
    SB_trials = V["sub"]["trials"]
    GT_meta = V["gt"]["meta"]
    SB_meta = V["sub"]["meta"]
    VM_COLS = V["vm_cols"]
    TRIAL_LEN = len(GT_trials[0]) if GT_trials else 100
    PATTERNS = ("00","01","10","11")
    RESP_LO_MS = int(V["behavior_params"]["RESP_WINDOW_LO_MS"])
    RESP_HI_MS = int(V["behavior_params"]["RESP_WINDOW_HI_MS"])

    # Peak detection parameters
    PEAK_PROMINENCE = config.get("psp_peak_prominence", 0.5)  # Minimum peak height
    MIN_PEAK_DISTANCE_MS = config.get("psp_min_peak_distance", 2)  # Minimum time between peaks
    BASELINE_PRE_MS = config.get("psp_baseline_pre", 10)  # Pre-stimulus baseline period
    CLIP_TO_RESP_WINDOW = config.get("psp_clip_to_window", True)  # Only count in response window

    def _earliest_anchor(m):
        """Get earliest stimulus time in trial"""
        a, b = m.get("a_time"), m.get("b_time")
        if a is None and b is None: return None
        return min([t for t in (a, b) if t is not None])

    def _resp_window_safe(m):
        """Get response window with fallback"""
        rw = m.get("response_window")
        if (rw is None) or (rw[1] < rw[0]):
            # Fallback to default window
            anc = _earliest_anchor(m)
            if anc is None:
                return 0, TRIAL_LEN - 1
            return max(0, anc + RESP_LO_MS), min(TRIAL_LEN - 1, anc + RESP_HI_MS)
        lo, hi = int(rw[0]), int(rw[1])
        return max(0, lo), min(TRIAL_LEN - 1, hi)

    def _baseline_subtracted(vec, m, pre_ms=BASELINE_PRE_MS):
        """Subtract baseline from VM trace"""
        v = np.asarray(vec, float)

        # Find baseline period before stimulus
        anc = _earliest_anchor(m)
        if anc is None:
            # No stimulus - use beginning of trial
            s, e = 0, min(TRIAL_LEN - 1, pre_ms)
        else:
            # Use period before stimulus
            s, e = max(0, int(anc - pre_ms)), max(0, int(anc - 1))

        # Calculate baseline as median of pre-stimulus period
        if e <= s:
            base = float(np.nanmedian(v[:max(1, pre_ms)]))
        else:
            base = float(np.nanmedian(v[s:e+1]))

        return v - base

    def _detect_psp_counts(v0, lo, hi):
        """Count PSP peaks in a VM trace segment"""
        v = np.asarray(v0, float)

        # Define analysis window
        use_lo = int(max(0, lo)) if CLIP_TO_RESP_WINDOW else 0
        use_hi = int(min(len(v)-1, hi)) if CLIP_TO_RESP_WINDOW else len(v)-1

        if use_hi < use_lo:
            return 0, 0

        seg = v[use_lo:use_hi+1]

        # Find positive peaks (EPSPs)
        p_up, _ = find_peaks(seg, prominence=PEAK_PROMINENCE,
                            distance=max(1, int(MIN_PEAK_DISTANCE_MS)))

        # Find negative peaks (IPSPs) by inverting signal
        p_down, _ = find_peaks(-seg, prominence=PEAK_PROMINENCE,
                              distance=max(1, int(MIN_PEAK_DISTANCE_MS)))

        return int(p_up.size), int(p_down.size)

    def _ids_by_pattern(meta, patt):
        """Get trial indices for a pattern"""
        return [i for i, m in enumerate(meta) if m.get("pattern") == patt]

    def _common_ids(patt):
        """Get common trial indices between GT and SUB"""
        return sorted(set(_ids_by_pattern(GT_meta, patt)).intersection(_ids_by_pattern(SB_meta, patt)))

    # Build PSP count tables for each pattern
    pattern_tables = {}

    for patt in PATTERNS:
        ids = _common_ids(patt)
        if not ids:
            continue

        rows_pattern = []
        for col in VM_COLS:
            # Initialize counters
            epsp_gt = ipsp_gt = epsp_sb = ipsp_sb = 0

            for i in ids:
                # Process GT trial
                m = GT_meta[i]
                lo, hi = _resp_window_safe(m)
                vbs = _baseline_subtracted(GT_trials[i][col].to_numpy(float), m)
                n_up, n_dn = _detect_psp_counts(vbs, lo, hi)
                epsp_gt += n_up
                ipsp_gt += n_dn

                # Process SUB trial
                m2 = SB_meta[i]
                lo2, hi2 = _resp_window_safe(m2)
                vbs2 = _baseline_subtracted(SB_trials[i][col].to_numpy(float), m2)
                n_up2, n_dn2 = _detect_psp_counts(vbs2, lo2, hi2)
                epsp_sb += n_up2
                ipsp_sb += n_dn2

            rows_pattern.append({
                "neuron": col.replace("_membrane_potential",""),
                "n_trials": len(ids),
                "EPSP_GT": epsp_gt,
                "EPSP_SUB": epsp_sb,
                "IPSP_GT": ipsp_gt,
                "IPSP_SUB": ipsp_sb
            })

        pattern_tables[patt] = pd.DataFrame(rows_pattern).sort_values(["neuron"]).reset_index(drop=True)

    return {"pattern_tables": pattern_tables}

# --- CROSS-CORRELATION METRIC ---
def compute_xcorr_metrics(V, config):
    """
    Compute cross-correlograms between neuron pairs.

    Cross-correlation reveals temporal relationships between neurons:
    - Peak at lag=0: Synchronous firing
    - Peak at positive lag: First neuron leads second
    - Peak at negative lag: Second neuron leads first

    Args:
        V: Preprocessed data structure
        config: Configuration parameters

    Returns:
        dict: Cross-correlation matrices and summary statistics
    """
    # Extract data
    GT_TRIALS = V["gt"]["trials"]
    SB_TRIALS = V["sub"]["trials"]
    GT_META = V["gt"]["meta"]
    SB_META = V["sub"]["meta"]
    SPIKE_COLS = list(V["spike_cols"])
    PATTERNS = ("00","01","10","11")

    RESP_LO_MS = int(V["behavior_params"]["RESP_WINDOW_LO_MS"])
    RESP_HI_MS = int(V["behavior_params"]["RESP_WINDOW_HI_MS"])
    MAX_LAG_MS = config.get("xcorr_max_lag", 15)  # Maximum lag to compute

    def _ids_by_pattern(meta, patt):
        """Get trial indices for a pattern"""
        return [i for i, m in enumerate(meta) if m.get("pattern") == patt]

    def _common_ids(patt):
        """Get common trial indices between GT and SUB"""
        return sorted(set(_ids_by_pattern(GT_META, patt)).intersection(_ids_by_pattern(SB_META, patt)))

    def _resp_window_ms(meta_row):
        """Get response window in milliseconds"""
        rw = meta_row.get("response_window")
        if rw is None:
            return int(RESP_LO_MS), int(RESP_HI_MS)
        lo, hi = int(rw[0]), int(rw[1])
        return max(0, lo), max(lo + 1, hi)

    def _resp_window_idx(df, meta_row):
        """Convert response window from ms to sample indices"""
        lo_ms, hi_ms = _resp_window_ms(meta_row)
        n = len(df)
        if n == 0:
            return 0, 0

        if "time" in df.columns:
            # Use actual time column if available
            t = df["time"].to_numpy(float)
            lo_idx = int(np.searchsorted(t, lo_ms, side="left"))
            hi_idx = int(np.searchsorted(t, hi_ms, side="right"))
        else:
            # Assume 1ms per sample
            lo_idx, hi_idx = lo_ms, hi_ms

        lo_idx = max(0, min(lo_idx, n))
        hi_idx = max(lo_idx + 1, min(hi_idx, n))
        return lo_idx, hi_idx

    def _seg_bin(df, col, lo_idx, hi_idx):
        """Extract binary spike train segment"""
        if (col not in df.columns) or lo_idx >= hi_idx:
            return np.zeros(0, dtype=np.int8)
        arr = df[col].to_numpy(int, copy=False)
        hi_idx = min(hi_idx, len(arr))
        return arr[lo_idx:hi_idx].astype(np.int8, copy=False)

    def _xcorr_norm(a, b, max_lag):
        """Compute normalized cross-correlation"""
        if a.size == 0 or b.size == 0:
            lags = np.arange(-max_lag, max_lag + 1, dtype=int)
            return lags, np.zeros(lags.size, dtype=float)

        W = int(min(a.size, b.size))
        if W <= 1:
            lags = np.arange(-max_lag, max_lag + 1, dtype=int)
            return lags, np.zeros_like(lags, dtype=float)

        L = int(min(max_lag, W - 1))

        # Full cross-correlation
        full = np.correlate(a.astype(float), b.astype(float), mode="full")
        l_full = np.arange(-(W - 1), (W - 1) + 1, dtype=int)

        # Select desired lag range
        sel = (l_full >= -L) & (l_full <= L)
        lags = l_full[sel]
        counts = full[sel].astype(float)

        # Normalize by effective window size
        eff = (W - np.abs(lags)).astype(float)
        eff[eff <= 0] = np.nan
        cc = counts / eff
        cc = np.where(np.isfinite(cc), cc, 0.0)

        return lags, cc

    def _avg_ccg_trials(trials_i, trials_j, meta, col_i, col_j, max_lag):
        """Average cross-correlogram across trials"""
        curves = []
        for df_i, df_j, m in zip(trials_i, trials_j, meta):
            lo, hi = _resp_window_idx(df_i, m)
            ai = _seg_bin(df_i, col_i, lo, hi)
            bj = _seg_bin(df_j, col_j, lo, hi)
            lags, cc = _xcorr_norm(ai, bj, max_lag)
            curves.append(cc)

        if not curves:
            lags = np.arange(-max_lag, max_lag + 1, dtype=int)
            mean_cc = np.zeros_like(lags, dtype=float)
        else:
            mean_cc = np.nanmean(np.vstack(curves), axis=0)

        return lags, mean_cc

    def _ccg_matrix_for_pattern(trials_block, meta_block, spike_cols, title, max_lag=MAX_LAG_MS):
        """Create cross-correlation matrix plot for all neuron pairs"""
        n = len(spike_cols)
        if n == 0 or not trials_block:
            return None, np.nan

        lags = np.arange(-max_lag, max_lag + 1, dtype=int)
        fig, axes = plt.subplots(n, n, figsize=(3.5 * n, 3.5 * n), sharex=True, sharey=True)
        if n == 1:
            axes = np.array([[axes]])

        zero_vals = []  # Store zero-lag values for summary

        for i, ci in enumerate(spike_cols):
            for j, cj in enumerate(spike_cols):
                # Compute average CCG for this neuron pair
                _, cc = _avg_ccg_trials(trials_block, trials_block, meta_block, ci, cj, max_lag)
                ax = axes[i, j]

                # Plot as bar chart
                ax.bar(lags, cc, width=1, color='tab:blue', alpha=0.7)
                ax.axvline(0, color="red", ls="--", lw=1)  # Mark zero lag

                # Format labels
                neuron_i = ci.replace("_spike_train", "")
                neuron_j = cj.replace("_spike_train", "")
                if i == n - 1:
                    ax.set_xlabel(f"Lag (ms)\n{neuron_j}", fontsize=9)
                if j == 0:
                    ax.set_ylabel(f"{neuron_i}\nCoincidence", fontsize=9)
                ax.tick_params(labelsize=8)

                # Collect zero-lag values for off-diagonal pairs
                if i != j:
                    zero_vals.append(cc[max_lag])

        fig.suptitle(title, fontsize=14)
        fig.tight_layout(rect=[0, 0, 1, 0.95])

        # Calculate mean zero-lag correlation (excluding auto-correlation)
        mean_zero = float(np.nan) if not zero_vals else float(np.nanmean(zero_vals))
        return fig, mean_zero

    # Special formatting for very small correlation values
    def _fmt_xcorr_val(x):
        """Format correlation values with appropriate precision"""
        if not np.isfinite(x):
            return np.nan
        if abs(x) < 0.001:
            # Show more precision for very small values
            return _fmt_decimal(x, 6)
        return _fmt_decimal(x, 3)

    # Generate plots and summary for each pattern
    rows, ccg_plots = [], []

    for patt in PATTERNS:
        ids = _common_ids(patt)
        if not ids:
            continue

        # Get trials for this pattern
        gt_trials_p = [GT_TRIALS[i] for i in ids]
        sb_trials_p = [SB_TRIALS[i] for i in ids]
        gt_meta_p = [GT_META[i] for i in ids]
        sb_meta_p = [SB_META[i] for i in ids]

        # Create CCG matrices for GT and SUB
        fig_gt, mean0_gt = _ccg_matrix_for_pattern(
            gt_trials_p, gt_meta_p, SPIKE_COLS,
            title=f"Cross Correlogram — Ground Truth — Pattern {patt}",
            max_lag=MAX_LAG_MS
        )
        fig_sb, mean0_sb = _ccg_matrix_for_pattern(
            sb_trials_p, sb_meta_p, SPIKE_COLS,
            title=f"Cross Correlogram — Submission — Pattern {patt}",
            max_lag=MAX_LAG_MS
        )

        # Save plots
        if fig_gt is not None:
            ccg_plots.append(_fig_to_png_bytes(fig_gt, dpi=150))
        if fig_sb is not None:
            ccg_plots.append(_fig_to_png_bytes(fig_sb, dpi=150))

        # Calculate difference in mean zero-lag correlation
        delta = (mean0_sb - mean0_gt) if (np.isfinite(mean0_gt) and np.isfinite(mean0_sb)) else np.nan

        rows.append(dict(
            Pattern=patt,
            MeanZeroLag_GT=_fmt_xcorr_val(mean0_gt),
            MeanZeroLag_SUB=_fmt_xcorr_val(mean0_sb),
            Delta=_fmt_xcorr_val(delta)
        ))

    ccg_summary = pd.DataFrame(rows, columns=["Pattern","MeanZeroLag_GT","MeanZeroLag_SUB","Delta"])

    return {
        "ccg_summary": ccg_summary,
        "ccg_plots": ccg_plots
    }

# --- VAN ROSSUM METRIC ---
def compute_vr_metrics(V, config):
    """
    Compute Van Rossum spike train distance metric.

    The Van Rossum distance measures dissimilarity between spike trains
    after convolving with an exponential kernel. It captures both
    spike count and timing differences.

    Args:
        V: Preprocessed data structure
        config: Configuration parameters

    Returns:
        dict: Van Rossum distances by pattern and neuron
    """
    # Extract data
    GT_TRIALS = V["gt"]["trials"]
    SB_TRIALS = V["sub"]["trials"]
    GT_META = V["gt"]["meta"]
    SB_META = V["sub"]["meta"]
    SPIKE_COLS = V["spike_cols"]
    PATTERNS = ("00","01","10","11")

    RESP_LO_MS = int(V.get("behavior_params", {}).get("RESP_WINDOW_LO_MS", 0))
    RESP_HI_MS = int(V.get("behavior_params", {}).get("RESP_WINDOW_HI_MS", 100))
    tau_ms = config.get("vr_tau_ms", 20.0)  # Time constant for exponential kernel

    def _ids_by_pattern(meta, patt):
        """Get trial indices for a pattern"""
        return [i for i, m in enumerate(meta) if m.get("pattern") == patt]

    def _common_ids(patt):
        """Get common trial indices between GT and SUB"""
        return sorted(set(_ids_by_pattern(GT_META, patt)).intersection(_ids_by_pattern(SB_META, patt)))

    def _resp_window(meta_row):
        """Get response window"""
        rw = meta_row.get("response_window")
        if rw is None:
            return RESP_LO_MS, RESP_HI_MS
        lo, hi = int(rw[0]), int(rw[1])
        return max(0, lo), max(lo+1, hi)

    def _spike_times_in_window(df, col, lo_ms, hi_ms):
        """Extract spike times within a window"""
        if df is None or df.empty or (col not in df.columns):
            return np.empty(0, dtype=float)

        arr = df[col].to_numpy(dtype=int)
        n = arr.size

        if "time" in df.columns:
            # Use actual time values
            t = df["time"].to_numpy(dtype=float)
            mask = (t >= lo_ms) & (t < hi_ms)
            idx = np.where((arr == 1) & mask)[0]
            return t[idx]
        else:
            # Assume 1ms per sample
            MS_PER_SAMPLE = 1.0
            lo = int(max(0, np.floor(lo_ms / MS_PER_SAMPLE)))
            hi = int(min(n, np.ceil(hi_ms / MS_PER_SAMPLE)))
            if lo >= hi:
                return np.empty(0, dtype=float)
            seg = arr[lo:hi]
            rel_idx = np.where(seg == 1)[0]
            return (lo + rel_idx).astype(float) * MS_PER_SAMPLE

    def _vr_distance(spike_t_x, spike_t_y, tau_ms):
        """
        Calculate Van Rossum distance between two spike trains.

        Based on: Van Rossum, M. C. (2001). A novel spike distance.
        Neural computation, 13(4), 751-763.
        """
        Nx = spike_t_x.size
        Ny = spike_t_y.size

        # Handle empty spike trains
        if Nx == 0 and Ny == 0:
            return 0.0
        if Nx == 0 or Ny == 0:
            return np.sqrt((Nx + Ny) / (2.0 * tau_ms))

        # Compute pairwise exponential similarities
        diffs = np.abs(spike_t_x[:, None] - spike_t_y[None, :])
        sxy = np.exp(-diffs / float(tau_ms)).sum()

        # Van Rossum distance formula
        d2 = (Nx + Ny - 2.0 * sxy) / (2.0 * float(tau_ms))
        return float(np.sqrt(max(d2, 0.0)))

    # Compute VR distance for each trial, pattern, and neuron
    records = []
    for patt in PATTERNS:
        ids = _common_ids(patt)
        if not ids:
            continue

        for i in ids:
            gt_df, sb_df = GT_TRIALS[i], SB_TRIALS[i]
            meta = GT_META[i] if i < len(GT_META) else {}
            lo_ms, hi_ms = _resp_window(meta)

            for neuron in SPIKE_COLS:
                # Get spike times for both datasets
                tx = _spike_times_in_window(gt_df, neuron, lo_ms, hi_ms)
                ty = _spike_times_in_window(sb_df, neuron, lo_ms, hi_ms)

                # Compute Van Rossum distance
                vr = _vr_distance(tx, ty, tau_ms=float(tau_ms))

                records.append({
                    "pattern": patt,
                    "trial_id": i,
                    "neuron": neuron.replace("_spike_train", ""),  # Short name
                    "VR": _fmt_decimal(vr, 3),
                    "GT_spikes": int(tx.size),
                    "SUB_spikes": int(ty.size),
                    "tau_ms": _fmt_decimal(tau_ms, 3),
                    "win_lo_ms": int(lo_ms),
                    "win_hi_ms": int(hi_ms),
                })

    trial_level_df = pd.DataFrame.from_records(records)

    if not trial_level_df.empty:
        # Summarize by pattern
        pattern_summary = (
            trial_level_df
            .groupby("pattern", as_index=False)
            .agg(n_trials=("trial_id", lambda x: len(np.unique(x))),
                 rows=("VR", "size"),
                 VR_mean=("VR", "mean"),
                 VR_median=("VR", "median"))
            .sort_values("pattern")
            .reset_index(drop=True)
        )

        # Summarize by neuron
        neuron_summary = (
            trial_level_df
            .groupby("neuron", as_index=False)
            .agg(rows=("VR","size"),
                 VR_mean=("VR","mean"),
                 VR_median=("VR","median"))
            .sort_values("neuron")
            .reset_index(drop=True)
        )

        # Format decimal values
        for col in ['VR_mean', 'VR_median']:
            if col in pattern_summary.columns:
                pattern_summary[col] = pattern_summary[col].apply(lambda x: _fmt_decimal(x, 3))
            if col in neuron_summary.columns:
                neuron_summary[col] = neuron_summary[col].apply(lambda x: _fmt_decimal(x, 3))
    else:
        # Empty summaries if no data
        pattern_summary = pd.DataFrame(columns=["pattern","n_trials","rows","VR_mean","VR_median"])
        neuron_summary = pd.DataFrame(columns=["neuron","rows","VR_mean","VR_median"])

    return {
        "vr_pattern_summary": pattern_summary,
        "vr_neuron_summary": neuron_summary
    }

# Continue with remaining metrics in next part...
# --- MULTI-SCALE CORRELATION (WITH FIXED OVERLAPPING) ---
def compute_msc_metrics(V, config):
    """
    Compute multi-scale correlation between spike trains.

    This metric evaluates spike train similarity at different temporal scales
    by convolving with Gaussian kernels of varying widths (sigma).
    Shows how correlation changes from fine to coarse temporal resolution.

    Args:
        V: Preprocessed data structure
        config: Configuration parameters

    Returns:
        dict: Multi-scale correlation curves and summaries
    """
    # Extract data
    GT_TRIALS = V["gt"]["trials"]
    SB_TRIALS = V["sub"]["trials"]
    GT_META = V["gt"]["meta"]
    SB_META = V["sub"]["meta"]
    PATTERNS = ("00","01","10","11")

    RESP_LO_MS = int(V.get("behavior_params", {}).get("RESP_WINDOW_LO_MS", 0))
    RESP_HI_MS = int(V.get("behavior_params", {}).get("RESP_WINDOW_HI_MS", 100))
    FS_HZ = float(V.get("fs_hz", 1000.0))
    MS_PER_SAMPLE = 1000.0 / FS_HZ if FS_HZ > 0 else 1.0

    # Get spike columns
    if "spike_cols" in V and V["spike_cols"]:
        SPIKE_COLS = list(V["spike_cols"])
    else:
        SPIKE_COLS = []

    # Return empty results if no data
    if not SPIKE_COLS or not GT_TRIALS or not SB_TRIALS:
        return {
            "msc_plots": [],
            "msc_pattern_summary": pd.DataFrame(),
            "msc_neuron_summary": pd.DataFrame()
        }

    # Range of smoothing scales to test (1ms to 100ms)
    sigma_ms = np.arange(1, 101)

    def _ids_by_pattern(meta, patt):
        """Get trial indices for a pattern"""
        return [i for i, m in enumerate(meta) if m.get("pattern") == patt]

    def _common_ids(patt):
        """Get common trial indices between GT and SUB"""
        return sorted(set(_ids_by_pattern(GT_META, patt)).intersection(_ids_by_pattern(SB_META, patt)))

    def _resp_window(meta_row):
        """Get response window"""
        rw = meta_row.get("response_window")
        if rw is None:
            return RESP_LO_MS, RESP_HI_MS
        lo, hi = int(rw[0]), int(rw[1])
        return max(0, lo), max(lo+1, hi)

    def _window_indices(df, lo_ms, hi_ms):
        """Convert time window to sample indices"""
        n = len(df)
        if n == 0:
            return 0, 0

        if "time" in df.columns:
            # Use actual time column
            t = df["time"].to_numpy(float)
            lo = int(np.searchsorted(t, lo_ms, side="left"))
            hi = int(np.searchsorted(t, hi_ms, side="left"))
            lo = max(0, min(lo, n))
            hi = max(0, min(hi, n))
            return lo, hi

        # Assume uniform sampling
        lo = int(max(0, math.floor(lo_ms / MS_PER_SAMPLE)))
        hi = int(min(n, math.ceil(hi_ms / MS_PER_SAMPLE)))
        return lo, hi

    def _get_bin_window(df, col, lo_idx, hi_idx):
        """Extract binary spike train segment"""
        if df is None or df.empty or (col not in df.columns):
            return np.zeros(0, dtype=float)

        arr = df[col].to_numpy(int)
        hi_idx = min(hi_idx, len(arr))
        if lo_idx >= hi_idx:
            return np.zeros(0, dtype=float)

        return arr[lo_idx:hi_idx].astype(float)

    def _gauss_kernel_sigma_samp(sig_samp):
        """Create Gaussian kernel with specified sigma in samples"""
        # Kernel size should cover ~6 sigma
        ksz = max(3, int(round(6.0 * sig_samp)))
        if ksz % 2 == 0:
            ksz += 1  # Keep kernel size odd

        # Create Gaussian kernel
        x = np.linspace(-3.0 * sig_samp, 3.0 * sig_samp, ksz)
        g = np.exp(-(x**2) / 2.0)
        g /= g.sum()  # Normalize

        return g

    def _pearson_r_safe(a, b):
        """Calculate Pearson correlation with safe handling of edge cases"""
        va = float(np.var(a))
        vb = float(np.var(b))

        # Handle constant signals
        if va == 0.0 and vb == 0.0:
            return 1.0  # Both constant and equal
        if va == 0.0 or vb == 0.0:
            return 0.0  # One is constant

        # Standard Pearson correlation
        ca = a - float(np.mean(a))
        cb = b - float(np.mean(b))
        denom = math.sqrt(float(np.sum(ca*ca)) * float(np.sum(cb*cb)))

        if denom == 0.0:
            return 0.0

        return float(np.sum(ca*cb) / denom)

    def _msc_curve_for_trial_neuron(gt_df, sb_df, neuron, lo_ms, hi_ms, sigma_vals_ms, dt_ms):
        """Compute correlation at multiple scales for one trial and neuron"""
        # Get spike trains
        lo_idx, hi_idx = _window_indices(gt_df, lo_ms, hi_ms)
        a = _get_bin_window(gt_df, neuron, lo_idx, hi_idx)
        b = _get_bin_window(sb_df, neuron, lo_idx, hi_idx)

        # Count spikes
        sa = int(a.sum())
        sb = int(b.sum())

        # Handle special cases
        if sa == 0 and sb == 0:
            # Both silent - perfect correlation
            return np.ones_like(sigma_vals_ms, dtype=float), True, False
        if (sa == 0 and sb > 0) or (sa > 0 and sb == 0):
            # Only one silent - no correlation
            return np.zeros_like(sigma_vals_ms, dtype=float), False, True

        # Compute correlation at each scale
        r = np.zeros_like(sigma_vals_ms, dtype=float)
        for k, sigma_ms in enumerate(sigma_vals_ms):
            # Convert sigma from ms to samples
            sig_samp = max(1e-6, float(sigma_ms / dt_ms))

            # Create Gaussian kernel
            g = _gauss_kernel_sigma_samp(sig_samp)

            # Convolve spike trains with kernel
            ca = np.convolve(a, g, mode="same")
            cb = np.convolve(b, g, mode="same")

            # Calculate correlation
            r[k] = _pearson_r_safe(ca, cb)

        return r, False, False

    # Storage for results
    per_pattern_per_neuron_curves = {p: {n: [] for n in SPIKE_COLS} for p in PATTERNS}
    all_records_neu = {n: [] for n in SPIKE_COLS}
    silent_both = {p: {n: 0 for n in SPIKE_COLS} for p in PATTERNS}
    silent_one = {p: {n: 0 for n in SPIKE_COLS} for p in PATTERNS}
    counts_trials = {p: 0 for p in PATTERNS}
    neuron_tot_trials = {n: 0 for n in SPIKE_COLS}
    neuron_both_silent = {n: 0 for n in SPIKE_COLS}
    neuron_one_silent = {n: 0 for n in SPIKE_COLS}

    # Process each pattern and trial
    for patt in PATTERNS:
        ids = _common_ids(patt)
        if not ids:
            continue
        counts_trials[patt] = len(ids)

        for i in ids:
            gt_df = GT_TRIALS[i]
            sb_df = SB_TRIALS[i]
            meta = GT_META[i] if i < len(GT_META) else {}
            lo_ms, hi_ms = _resp_window(meta)

            for neu in SPIKE_COLS:
                neuron_tot_trials[neu] += 1

                # Compute multi-scale correlation curve
                r_vec, is_both_silent, is_one_silent = _msc_curve_for_trial_neuron(
                    gt_df, sb_df, neu, lo_ms, hi_ms, sigma_ms, MS_PER_SAMPLE
                )

                # Store results
                per_pattern_per_neuron_curves[patt][neu].append(r_vec)
                all_records_neu[neu].extend(r_vec.tolist())

                # Track silent cases
                if is_both_silent:
                    silent_both[patt][neu] += 1
                    neuron_both_silent[neu] += 1
                if is_one_silent:
                    silent_one[patt][neu] += 1
                    neuron_one_silent[neu] += 1

    # Create pattern summary
    rows_patt = []
    for patt in PATTERNS:
        vals = []
        bs = os_ = 0
        tot_trials_for_pattern = counts_trials[patt] * len(SPIKE_COLS) if counts_trials[patt] > 0 else 0

        if counts_trials[patt] > 0:
            for neu in SPIKE_COLS:
                curves = per_pattern_per_neuron_curves[patt][neu]
                if curves:
                    vals.extend(np.concatenate(curves).tolist())
                    bs += silent_both[patt][neu]
                    os_ += silent_one[patt][neu]

        if vals:
            arr = np.asarray(vals, dtype=float)
            rows_patt.append({
                "pattern": patt,
                "n_trials": counts_trials[patt],
                "rows": int(arr.size),
                "r_mean": _fmt_decimal(np.mean(arr), 3),
                "r_median": _fmt_decimal(np.median(arr), 3),
                "both_silent_pct": _fmt_decimal(100.0 * (bs / tot_trials_for_pattern) if tot_trials_for_pattern > 0 else np.nan, 1),
                "one_silent_pct": _fmt_decimal(100.0 * (os_ / tot_trials_for_pattern) if tot_trials_for_pattern > 0 else np.nan, 1)
            })
    pattern_summary = pd.DataFrame(rows_patt)

    # Create neuron summary
    rows_neu = []
    for neu in SPIKE_COLS:
        vals = np.asarray(all_records_neu[neu], dtype=float)
        tot_trials = neuron_tot_trials[neu]

        if vals.size > 0 and tot_trials > 0:
            rows_neu.append({
                "neuron": neu.replace("_spike_train", ""),  # Short name
                "rows": int(vals.size),
                "r_mean": _fmt_decimal(np.mean(vals), 3),
                "r_median": _fmt_decimal(np.median(vals), 3),
                "both_silent_pct": _fmt_decimal(100.0 * neuron_both_silent[neu] / tot_trials, 1),
                "one_silent_pct": _fmt_decimal(100.0 * neuron_one_silent[neu] / tot_trials, 1),
            })
    neuron_summary = pd.DataFrame(rows_neu)

    # Generate plots showing correlation vs smoothing scale
    msc_plots = []
    for patt in PATTERNS:
        ids = _common_ids(patt)
        if not ids:
            continue

        fig, ax = plt.subplots(figsize=(9, 5))

        # Plot mean curve for each neuron
        for neu in SPIKE_COLS:
            curves = per_pattern_per_neuron_curves[patt][neu]
            if not curves:
                continue

            # Stack curves and compute mean
            C = np.vstack(curves)
            mean_curve = np.mean(C, axis=0)

            # Plot with neuron label
            label = neu.replace("_spike_train", "")  # Short name
            ax.plot(sigma_ms, mean_curve, lw=1.8, label=label)

        ax.set_title(f"Multi-Scale Correlation — Pattern {patt}")
        ax.set_xlabel("σ (ms)")
        ax.set_ylabel("Correlation r")
        ax.set_ylim(-0.05, 1.05)
        ax.legend(loc="best", ncol=2 if len(SPIKE_COLS) > 4 else 1, fontsize=9)
        ax.grid(alpha=0.25)
        plt.tight_layout()
        msc_plots.append(_fig_to_png_bytes(fig, dpi=150))
        plt.close()

    return {
        "msc_plots": msc_plots,
        "msc_pattern_summary": pattern_summary,
        "msc_neuron_summary": neuron_summary
    }

# --- SCHREIBER METRIC (WITH FIXED NAN HANDLING) ---
def compute_schreiber_metrics(V, config):
    """
    Compute Schreiber similarity metric for spike trains.

    The Schreiber similarity is a correlation-based measure after
    convolving spike trains with a Gaussian kernel. Unlike multi-scale
    correlation, this uses a fixed sigma value.

    Reference: Schreiber, S., et al. (2003). A new correlation-based
    measure of spike timing reliability. Neurocomputing, 52, 925-931.

    Args:
        V: Preprocessed data structure
        config: Configuration parameters

    Returns:
        dict: Schreiber similarity values and plots
    """
    # Extract data
    GT_TRIALS = V["gt"]["trials"]
    SB_TRIALS = V["sub"]["trials"]
    GT_META = V["gt"]["meta"]
    SB_META = V["sub"]["meta"]
    PATTERNS = ("00","01","10","11")

    RESP_LO_MS = int(V.get("behavior_params", {}).get("RESP_WINDOW_LO_MS", 0))
    RESP_HI_MS = int(V.get("behavior_params", {}).get("RESP_WINDOW_HI_MS", 100))
    FS_HZ = float(V.get("fs_hz", 1000.0))
    MS_PER_SAMPLE = 1000.0 / FS_HZ if FS_HZ > 0 else 1.0

    # Get spike columns
    if "spike_cols" in V and V["spike_cols"]:
        SPIKE_COLS = list(V["spike_cols"])
    else:
        SPIKE_COLS = []

    # Return empty results if no data
    if not SPIKE_COLS or not GT_TRIALS or not SB_TRIALS:
        return {
            "schreiber_plots": [],
            "schreiber_pattern_summary": pd.DataFrame(),
            "schreiber_neuron_summary": pd.DataFrame()
        }

    # Fixed smoothing scale
    sigma_ms = config.get("schreiber_sigma_ms", 10.0)

    def _ids_by_pattern(meta, patt):
        """Get trial indices for a pattern"""
        return [i for i, m in enumerate(meta) if m.get("pattern") == patt]

    def _common_ids(patt):
        """Get common trial indices between GT and SUB"""
        return sorted(set(_ids_by_pattern(GT_META, patt)).intersection(_ids_by_pattern(SB_META, patt)))

    def _resp_window(meta_row):
        """Get response window"""
        rw = meta_row.get("response_window")
        if rw is None:
            return RESP_LO_MS, RESP_HI_MS
        lo, hi = int(rw[0]), int(rw[1])
        return max(0, lo), max(lo+1, hi)

    def _binary_segment_in_window(df, col, lo_ms, hi_ms):
        """Extract binary spike train segment within window"""
        if df is None or df.empty or (col not in df.columns):
            return np.zeros(0, dtype=np.float32)

        x = df[col].to_numpy(dtype=np.float32)
        n = x.size

        if "time" in df.columns:
            # Use actual time column
            t = df["time"].to_numpy(dtype=float)
            mask = (t >= lo_ms) & (t < hi_ms)
            if not np.any(mask):
                return np.zeros(0, dtype=np.float32)
            return x[mask]
        else:
            # Assume uniform sampling
            lo = int(max(0, np.floor(lo_ms / MS_PER_SAMPLE)))
            hi = int(min(n, np.ceil(hi_ms / MS_PER_SAMPLE)))
            if lo >= hi:
                return np.zeros(0, dtype=np.float32)
            return x[lo:hi]

    def _gaussian_kernel(sigma_ms, fs_hz):
        """Create Gaussian kernel for smoothing"""
        dt = 1000.0 / float(fs_hz)  # ms per sample
        sigma = float(sigma_ms)

        if sigma <= 0:
            return np.array([1.0], dtype=np.float32)

        # Kernel size to cover ~6 sigma
        ksz = max(3, int(round(6.0 * sigma / dt)))
        if ksz % 2 == 0:
            ksz += 1  # Keep odd size

        half = ksz // 2
        t = (np.arange(ksz) - half) * dt
        g = np.exp(-0.5 * (t / sigma) ** 2).astype(np.float32)

        # Normalize
        s = float(g.sum())
        if s > 0:
            g /= s

        return g

    def _schreiber_similarity(a_bin, b_bin, kernel):
        """
        Calculate Schreiber similarity between two spike trains.

        Returns correlation after convolution with kernel.
        """
        # Check if either train has spikes
        na = int(a_bin.sum() > 0)
        nb = int(b_bin.sum() > 0)
        both_silent = (na == 0 and nb == 0)
        one_silent = ((na == 0) ^ (nb == 0))

        if a_bin.size == 0 or b_bin.size == 0:
            return np.nan, both_silent, one_silent

        # Convolve with kernel
        ca = np.convolve(a_bin, kernel, mode="same")
        cb = np.convolve(b_bin, kernel, mode="same")

        # Calculate correlation
        num = float(np.dot(ca, cb))
        den = float(np.sqrt(np.dot(ca, ca) * np.dot(cb, cb)))

        if den == 0.0:
            return np.nan, both_silent, one_silent

        r = num / den
        # Clip to valid correlation range
        if r < -1.0: r = -1.0
        if r > 1.0: r = 1.0

        return r, both_silent, one_silent

    # Compute Schreiber similarity for all trials
    kernel = _gaussian_kernel(sigma_ms=float(sigma_ms), fs_hz=FS_HZ)

    rows = []
    for patt in PATTERNS:
        ids = _common_ids(patt)
        if not ids:
            continue

        for i in ids:
            gt_df = GT_TRIALS[i]
            sb_df = SB_TRIALS[i]
            meta = GT_META[i] if i < len(GT_META) else {}
            lo_ms, hi_ms = _resp_window(meta)

            for neuron in SPIKE_COLS:
                # Get spike trains
                a = _binary_segment_in_window(gt_df, neuron, lo_ms, hi_ms)
                b = _binary_segment_in_window(sb_df, neuron, lo_ms, hi_ms)

                # Calculate similarity
                r, both_silent, one_silent = _schreiber_similarity(a, b, kernel)

                rows.append({
                    "pattern": patt,
                    "trial_id": i,
                    "neuron": neuron,
                    "r": _fmt_decimal(r, 3) if (r == r) else np.nan,  # NaN check
                    "both_silent": bool(both_silent),
                    "one_silent": bool(one_silent),
                    "win_lo_ms": int(lo_ms),
                    "win_hi_ms": int(hi_ms),
                    "sigma_ms": _fmt_decimal(sigma_ms, 3),
                })

    trial_level = pd.DataFrame.from_records(rows)

    if trial_level.empty:
        return {
            "schreiber_plots": [],
            "schreiber_pattern_summary": pd.DataFrame(),
            "schreiber_neuron_summary": pd.DataFrame()
        }

    # Helper functions for safe NaN handling
    def safe_nanmean(x):
        """Calculate mean ignoring NaN values"""
        vals = [v for v in x if np.isfinite(v)]
        return _fmt_decimal(np.mean(vals), 3) if vals else np.nan

    def safe_nanmedian(x):
        """Calculate median ignoring NaN values"""
        vals = [v for v in x if np.isfinite(v)]
        return _fmt_decimal(np.median(vals), 3) if vals else np.nan

    # Create pattern summary
    pattern_summary = (
        trial_level
        .groupby("pattern", as_index=False)
        .agg(
            n_trials=("trial_id", lambda x: len(np.unique(x))),
            rows=("r", "size"),
            r_mean=("r", safe_nanmean),
            r_median=("r", safe_nanmedian),
            masked_pct=("r", lambda x: _fmt_decimal(float(np.mean(np.isnan(x))) * 100.0, 1))
        )
        .sort_values("pattern")
        .reset_index(drop=True)
    )

    # Create neuron summary
    neuron_summary = (
        trial_level
        .groupby("neuron", as_index=False)
        .agg(
            rows=("r", "size"),
            r_mean=("r", safe_nanmean),
            r_median=("r", safe_nanmedian),
            both_silent_pct=("both_silent", lambda x: _fmt_decimal(float(np.mean(x)) * 100.0, 1)),
            one_silent_pct=("one_silent", lambda x: _fmt_decimal(float(np.mean(x)) * 100.0, 1)),
        )
        .sort_values("neuron")
        .reset_index(drop=True)
    )

    # Shorten neuron names in summary
    neuron_summary['neuron'] = neuron_summary['neuron'].str.replace('_spike_train', '')

    # Generate plots for each pattern
    schreiber_plots = []
    for patt in PATTERNS:
        dfp = trial_level.loc[trial_level["pattern"] == patt]
        if dfp.empty:
            continue

        # Aggregate by neuron
        g = (dfp.groupby("neuron", as_index=False)
                .agg(r_median=("r","median"),
                     usable_frac=("both_silent", lambda x: 1.0 - float(np.mean(x)) if len(x) else np.nan)))

        # Shorten neuron names for plot
        g['neuron'] = g['neuron'].str.replace('_spike_train', '')

        # Maintain neuron order
        neuron_order = [n.replace('_spike_train', '') for n in SPIKE_COLS]
        g = g.set_index("neuron").reindex(neuron_order)
        no_data_mask = g["usable_frac"].isna() & g["r_median"].isna()
        g["usable_frac"] = g["usable_frac"].fillna(0.0)
        g = g.reset_index()
        x = np.arange(len(g))

        # Create dual-axis plot
        fig, ax1 = plt.subplots(figsize=(max(8, 1.8 + 0.8*len(g)), 5))
        ax2 = ax1.twinx()

        # Bar chart for usable fraction
        bars = ax2.bar(g["neuron"], g["usable_frac"].to_numpy(float), alpha=0.28, width=0.8,
                       label="Usable frac", zorder=1, color='lightblue')

        # Mark neurons with no data
        if no_data_mask.any():
            for xi in x[no_data_mask.to_numpy()]:
                ax2.bar(g["neuron"].iloc[xi], 0.02, fill=False, hatch="///",
                       edgecolor="k", linewidth=1.0, alpha=0.9, zorder=3)

        ax2.set_ylim(0.0, 1.0)
        ax2.set_ylabel("Usable fraction")

        # Line plot for median similarity
        ax1.plot(g["neuron"], g["r_median"].to_numpy(float), marker="o", linewidth=1.8,
                 label="Median similarity", zorder=4, color='darkblue')
        ax1.set_ylim(0.0, 1.0)
        ax1.set_ylabel("Median Schreiber r")
        ax1.set_xlabel("Neuron")
        ax1.set_title(f"Schreiber (σ={sigma_ms} ms) — Pattern {patt}")
        ax1.grid(alpha=0.25, axis="y")

        # Fix overlapping x-axis labels
        plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')

        # Combined legend
        h1, l1 = ax1.get_legend_handles_labels()
        h2, l2 = ax2.get_legend_handles_labels()
        if no_data_mask.any():
            from matplotlib.patches import Patch
            h2.append(Patch(fill=False, hatch="///", edgecolor="k", linewidth=1.0))
            l2.append("No data")
        ax1.legend(h1+h2, l1+l2, loc="lower right", fontsize=9)

        plt.tight_layout()
        schreiber_plots.append(_fig_to_png_bytes(fig, dpi=150))
        plt.close()

    return {
        "schreiber_plots": schreiber_plots,
        "schreiber_pattern_summary": pattern_summary,
        "schreiber_neuron_summary": neuron_summary
    }

# Continue with remaining metrics...
# --- GRANGER CAUSALITY (WITH PROPER CHART SIZING) ---
def compute_granger_metrics(V, config):
    """
    Compute Granger causality analysis between neurons.

    Granger causality tests whether past values of one time series help
    predict future values of another. If neuron X Granger-causes neuron Y,
    then X's past activity improves prediction of Y's future activity.

    Uses autoregressive models and F-tests to detect causal relationships.

    Args:
        V: Preprocessed data structure
        config: Configuration parameters

    Returns:
        dict: Granger causality matrices and network summaries
    """
    # Extract data
    GT_TRIALS = V["gt"]["trials"]
    SB_TRIALS = V["sub"]["trials"]
    GT_META = V["gt"]["meta"]
    SB_META = V["sub"]["meta"]
    PATTERNS = ("00","01","10","11")

    RESP_LO_MS = int(V.get("behavior_params", {}).get("RESP_WINDOW_LO_MS", 0))
    RESP_HI_MS = int(V.get("behavior_params", {}).get("RESP_WINDOW_HI_MS", 100))
    FS_HZ = float(V.get("fs_hz", 1000.0))
    MS_PER_SAMPLE = 1000.0 / FS_HZ if FS_HZ > 0 else 1.0

    # Get spike columns
    if "spike_cols" in V and V["spike_cols"]:
        SPIKE_COLS = list(V["spike_cols"])
    else:
        SPIKE_COLS = []

    # Return empty results if no data
    if not SPIKE_COLS or not GT_TRIALS or not SB_TRIALS:
        return {
            "granger_plots": [],
            "granger_summary": pd.DataFrame()
        }

    # Granger analysis parameters
    bin_ms = config.get("granger_bin_ms", 5)  # Time bin size for discretization
    lag_bins = config.get("granger_lag_bins", 10)  # Number of past bins to use in model
    alpha = config.get("granger_alpha", 0.05)  # Significance level
    min_bins = config.get("granger_min_bins", 50)  # Minimum bins needed for analysis

    # Check if SciPy is available for F-distribution
    try:
        from scipy.stats import f as _f_dist
        _HAS_SCIPY = True
    except:
        _HAS_SCIPY = False

    def _ids_by_pattern(meta, patt):
        """Get trial indices for a pattern"""
        return [i for i, m in enumerate(meta) if m.get("pattern") == patt]

    def _common_ids(patt):
        """Get common trial indices between GT and SUB"""
        return sorted(set(_ids_by_pattern(GT_META, patt)).intersection(_ids_by_pattern(SB_META, patt)))

    def _resp_window(meta_row):
        """Get response window"""
        rw = meta_row.get("response_window")
        if rw is None:
            return RESP_LO_MS, RESP_HI_MS
        lo, hi = int(rw[0]), int(rw[1])
        return max(0, lo), max(lo+1, hi)

    def _slice_window(df, lo_ms, hi_ms):
        """Create boolean mask for time window"""
        n = len(df)
        if "time" in df.columns:
            t = df["time"].to_numpy(float)
            return (t >= lo_ms) & (t < hi_ms)

        # Assume uniform sampling
        lo = int(max(0, np.floor(lo_ms / MS_PER_SAMPLE)))
        hi = int(min(n, np.ceil(hi_ms / MS_PER_SAMPLE)))
        mask = np.zeros(n, dtype=bool)
        if lo < hi:
            mask[lo:hi] = True
        return mask

    def _bin_spikes_block(trials, metas, spike_cols, bin_ms=5):
        """
        Bin spike trains across multiple trials.

        Converts high-resolution spike trains into binned counts
        for time series analysis.
        """
        if not trials or not spike_cols:
            return np.zeros((0,0)), np.array([])

        pieces = []
        for df, meta in zip(trials, metas):
            # Get response window
            lo_ms, hi_ms = _resp_window(meta)
            mask = _slice_window(df, lo_ms, hi_ms)
            if not mask.any():
                continue

            # Extract spike data
            block = df.loc[mask, spike_cols].to_numpy(int)
            W = block.shape[0]

            if "time" in df.columns:
                # Use actual time values for binning
                t = df.loc[mask, "time"].to_numpy(float)
                start = t[0]; end = t[-1]
                edges = np.arange(start, end+1e-6, bin_ms)
                idx = np.searchsorted(edges, t, side="right") - 1
                B = len(edges)-1

                # Count spikes in each bin
                binned = np.zeros((B, block.shape[1]), dtype=float)
                for b in range(B):
                    sel = (idx == b)
                    if sel.any():
                        binned[b] = block[sel].sum(axis=0)
            else:
                # Assume uniform sampling
                step_ms = MS_PER_SAMPLE
                B = int(np.ceil(W * step_ms / bin_ms))
                binned = np.zeros((B, block.shape[1]), dtype=float)

                for b in range(B):
                    lo = int(round((b * bin_ms) / step_ms))
                    hi = int(round(((b+1) * bin_ms) / step_ms))
                    hi = min(hi, W)
                    if lo < hi:
                        binned[b] = block[lo:hi].sum(axis=0)

            pieces.append(binned)

        if not pieces:
            return np.zeros((0, len(spike_cols))), np.array([])

        # Concatenate all trials
        X = np.vstack(pieces)
        edges_ms = np.arange(X.shape[0]+1) * bin_ms
        return X, edges_ms

    def _lag_design(y, X_lags, lag):
        """
        Create design matrices for autoregressive model.

        Builds restricted model (only Y's past) and full model
        (Y's past + X's past) for Granger causality test.
        """
        T = y.shape[0]
        if T <= lag:
            return np.zeros(0), np.zeros((0, lag+1)), np.zeros((0, 2*lag+1))

        rows = T - lag
        Y = y[lag:]  # Target: future values of Y

        # Create lagged versions
        def _lags(v):
            """Create matrix of lagged values"""
            return np.column_stack([v[lag-k-1:T-k-1] for k in range(lag)])

        Ylags = _lags(y)  # Past values of Y
        Xlags = _lags(X_lags)  # Past values of X

        # Restricted model: Y predicted from its own past only
        R = np.column_stack([np.ones(rows), Ylags])

        # Full model: Y predicted from both Y's and X's past
        F = np.column_stack([R, Xlags])

        return Y, R, F

    def _ols_rss(design, target):
        """
        Ordinary least squares residual sum of squares.

        Fits linear model and returns sum of squared residuals.
        """
        if design.shape[0] == 0 or design.shape[1] == 0:
            return np.nan

        # Solve least squares problem
        beta, *_ = np.linalg.lstsq(design, target, rcond=None)
        resid = target - design @ beta
        return float(np.dot(resid, resid))

    def _granger_pair(y, x, lag):
        """
        Test if X Granger-causes Y.

        Compares predictive power of restricted vs full model.
        """
        Y, R, F = _lag_design(y, x, lag)
        if Y.size == 0:
            return dict(effect=np.nan, p=np.nan)

        # Fit both models
        rss_r = _ols_rss(R, Y)  # Restricted model RSS
        rss_f = _ols_rss(F, Y)  # Full model RSS

        if not np.isfinite(rss_r) or not np.isfinite(rss_f) or rss_f <= 0:
            return dict(effect=np.nan, p=np.nan)

        # Effect size: log ratio of RSS
        effect = np.log(rss_r / rss_f)

        # Statistical test using F-distribution
        if _HAS_SCIPY:
            df1 = F.shape[1] - R.shape[1]  # Extra parameters in full model
            df2 = F.shape[0] - F.shape[1]  # Residual degrees of freedom

            if df1 > 0 and df2 > 0:
                # F-statistic
                Fstat = ((rss_r - rss_f) / df1) / (rss_f / df2)
                # p-value from F-distribution
                p = _f_dist.sf(Fstat, df1, df2)
            else:
                p = np.nan
        else:
            p = np.nan

        return dict(effect=float(effect), p=float(p) if np.isfinite(p) else np.nan)

    def _bh_fdr(pvals, alpha=0.05):
        """
        Benjamini-Hochberg FDR correction for multiple comparisons.

        Controls false discovery rate when testing multiple hypotheses.
        """
        p = np.asarray(pvals, float)
        m = np.sum(np.isfinite(p))
        if m == 0:
            return np.full_like(p, np.nan)

        # Sort p-values
        order = np.argsort(np.where(np.isfinite(p), p, np.inf))
        ranks = np.empty_like(order)
        ranks[order] = np.arange(1, len(p)+1)

        # Calculate adjusted p-values
        q = np.full_like(p, np.nan)
        q_work = np.where(np.isfinite(p), p * m / ranks, np.nan)

        # Enforce monotonicity
        prev = np.inf
        for idx in order[::-1]:
            if np.isfinite(q_work[idx]):
                prev = min(prev, q_work[idx])
                q[idx] = prev

        return q

    def _gc_for_dataset(trials, metas, spike_cols, bin_ms=5, lag_bins=10,
                        alpha=0.05, min_bins=50, min_spike_sum=10):
        """
        Compute Granger causality matrix for entire dataset.

        Returns matrices of effect sizes, p-values, and significant edges.
        """
        # Bin spike trains
        X, _ = _bin_spikes_block(trials, metas, spike_cols, bin_ms=bin_ms)
        N = len(spike_cols)

        # Check if we have enough data
        if X.shape[0] < max(min_bins, lag_bins+5):
            return (np.full((N,N), np.nan),)*3 + (np.zeros((N,N), bool),)

        # Check which neurons have enough spikes
        spike_sums = X.sum(axis=0)
        ok_neuron = spike_sums >= min_spike_sum

        # Z-score normalize each neuron's activity
        Xz = X.copy().astype(float)
        for j in range(N):
            if ok_neuron[j]:
                mu = Xz[:,j].mean()
                sd = Xz[:,j].std(ddof=1)
                Xz[:,j] = (Xz[:,j]-mu) / (sd if sd>0 else 1.0)
            else:
                Xz[:,j] = 0.0

        # Initialize result matrices
        M_eff = np.full((N,N), np.nan)  # Effect sizes
        M_p = np.full((N,N), np.nan)    # p-values

        # Test all neuron pairs
        for j in range(N):
            y = Xz[:, j]
            if not ok_neuron[j]:
                continue

            for i in range(N):
                if i == j or not ok_neuron[i]:
                    continue

                # Test if neuron i causes neuron j
                res = _granger_pair(y, Xz[:, i], lag_bins)
                M_eff[j, i] = res["effect"]
                M_p[j, i] = res["p"]

        # Apply FDR correction
        q = _bh_fdr(M_p.ravel(), alpha=alpha).reshape(M_p.shape)

        # Mark significant edges
        usable = np.isfinite(M_eff) & np.isfinite(q) & (q <= alpha)

        return M_eff, M_p, q, usable

    # Process each pattern
    rows = []
    granger_plots = []

    for patt in PATTERNS:
        ids = _common_ids(patt)
        if not ids:
            continue

        # Get trials for this pattern
        gt_tr = [GT_TRIALS[i] for i in ids]
        sb_tr = [SB_TRIALS[i] for i in ids]
        gt_mt = [GT_META[i] for i in ids]
        sb_mt = [SB_META[i] for i in ids]

        # Compute Granger causality for GT and SUB
        eff_gt, p_gt, q_gt, use_gt = _gc_for_dataset(
            gt_tr, gt_mt, SPIKE_COLS, bin_ms=bin_ms,
            lag_bins=lag_bins, alpha=alpha
        )
        eff_sb, p_sb, q_sb, use_sb = _gc_for_dataset(
            sb_tr, sb_mt, SPIKE_COLS, bin_ms=bin_ms,
            lag_bins=lag_bins, alpha=alpha
        )

        # Count edges (excluding self-connections)
        A = use_gt
        B = use_sb
        e_gt = int(A.sum() - np.trace(A))
        e_sb = int(B.sum() - np.trace(B))
        inter = int((A & B).sum() - np.trace(A & B))  # Overlapping edges
        union = int((A | B).sum() - np.trace(A | B))  # Total unique edges
        jacc = _fmt_decimal((inter / union) if union > 0 else np.nan, 3)

        rows.append(dict(
            pattern=patt,
            edges_GT=e_gt,
            edges_SUB=e_sb,
            overlap=inter,
            jaccard=jacc
        ))

        # Create visualization functions
        def _plot_heat(ax, M, mask, title):
            """Plot Granger causality heatmap"""
            im = ax.imshow(np.where(mask, M, np.nan), cmap="magma", aspect="equal")
            ax.set_title(title, fontsize=10)

            # Format labels
            labels = [c.replace("_spike_train", "") for c in SPIKE_COLS]
            ax.set_xticks(np.arange(len(SPIKE_COLS)))
            ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
            ax.set_yticks(np.arange(len(SPIKE_COLS)))
            ax.set_yticklabels(labels, fontsize=8)

            # Add colorbar
            cb = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            cb.set_label("GC effect", fontsize=8)
            cb.ax.tick_params(labelsize=7)

        def _plot_degree(ax, mask, title):
            """Plot in-degree and out-degree distributions"""
            outdeg = mask.sum(axis=0)  # Number of neurons this neuron influences
            indeg = mask.sum(axis=1)   # Number of neurons influencing this neuron

            x = np.arange(len(SPIKE_COLS))
            labels = [c.replace("_spike_train", "") for c in SPIKE_COLS]

            # Bar plot
            ax.bar(x-0.2, outdeg, width=0.4, label="out-degree", color='tab:blue')
            ax.bar(x+0.2, indeg, width=0.4, label="in-degree", color='tab:orange')
            ax.set_xticks(x)
            ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
            ax.set_title(title, fontsize=10)
            ax.legend(fontsize=8)
            ax.grid(axis='y', alpha=0.25)

        # Create figure with three panels
        fig = plt.figure(figsize=(14, 4.5))  # Compact height
        gs = fig.add_gridspec(1, 3, width_ratios=[1, 1, 1.2])
        ax1 = fig.add_subplot(gs[0])
        ax2 = fig.add_subplot(gs[1])
        ax3 = fig.add_subplot(gs[2])

        _plot_heat(ax1, eff_gt, use_gt, f"GT — Pattern {patt}")
        _plot_heat(ax2, eff_sb, use_sb, f"SUB — Pattern {patt}")
        _plot_degree(ax3, use_sb, f"Degree (q≤{alpha}) — SUB")

        fig.suptitle(f"Granger Causality (bin={bin_ms} ms, lag={lag_bins} bins) — Pattern {patt}",
                     fontsize=12)
        fig.tight_layout(rect=[0,0,1,0.94])
        granger_plots.append(_fig_to_png_bytes(fig, dpi=150))
        plt.close()

    summary = pd.DataFrame(rows, columns=["pattern","edges_GT","edges_SUB","overlap","jaccard"])

    return {
        "granger_plots": granger_plots,
        "granger_summary": summary
    }

# --- BEHAVIORAL XOR METRIC (WITH CORRECT PATTERN ORDERING) ---
def compute_behavioral_metrics(V, config):
    """
    Evaluate XOR logic behavioral performance.

    Tests whether the network correctly implements XOR logic:
    - A=0, B=0 → E=0
    - A=1, B=0 → E=1
    - A=0, B=1 → E=1
    - A=1, B=1 → E=0

    Measures accuracy, sensitivity, and specificity of E neuron responses.

    Args:
        V: Preprocessed data structure
        config: Configuration parameters

    Returns:
        dict: Truth table and performance metrics
    """
    # Extract data
    GT_TRIALS = V["gt"]["trials"]
    SB_TRIALS = V["sub"]["trials"]
    GT_META = V["gt"]["meta"]
    SB_META = V["sub"]["meta"]

    # Map output neuron E
    OUT_MAP = V.get("outputs_map", {"E":"E_spike_train"})
    E_COL = OUT_MAP.get("E", "E_spike_train")

    # XOR truth table: expected E output for each pattern
    XOR_E = {"00": 0,  # No inputs → No output
             "01": 1,  # B only → Output
             "10": 1,  # A only → Output
             "11": 0}  # Both inputs → No output (XOR suppression)

    def _score_side(trials, meta, label):
        """
        Score behavioral performance for one dataset (GT or SUB).

        Calculates confusion matrix and performance metrics.
        """
        # Initialize confusion matrix for each pattern
        agg = {p: {"TP": 0, "FN": 0, "TN": 0, "FP": 0}
               for p in ["00", "01", "10", "11"]}

        for df, m in zip(trials, meta):
            pattern = str(m.get("pattern", ""))
            if pattern not in agg:
                continue

            # Determine response window
            if m.get("response_window"):
                lo, hi = m["response_window"]
                lo = max(0, int(lo))
                hi = min(99, int(hi))
            else:
                # Default window if not specified
                lo, hi = 5, 95

            # Get expected output for this pattern
            want = XOR_E[pattern]

            # Check if E neuron fired in response window
            if E_COL in df.columns and lo <= hi:
                spike_vec = df[E_COL].values[lo:hi+1]
                have = int(np.any(spike_vec > 0))  # Did E fire at all?
            else:
                have = 0

            # Update confusion matrix
            if want == 1 and have == 1:
                agg[pattern]["TP"] += 1  # True Positive
            elif want == 1 and have == 0:
                agg[pattern]["FN"] += 1  # False Negative
            elif want == 0 and have == 0:
                agg[pattern]["TN"] += 1  # True Negative
            else:
                agg[pattern]["FP"] += 1  # False Positive

        # Build performance table with CORRECT pattern ordering
        rows = []

        # Pattern naming for display
        # NOTE: The ordering is crucial for matching expected results
        pattern_names = {
            "00": "Pattern 1 (A=0,B=0)",
            "10": "Pattern 2 (A=1,B=0)",  # Pattern 2 is "10" not "01"!
            "01": "Pattern 3 (A=0,B=1)",  # Pattern 3 is "01" not "10"!
            "11": "Pattern 4 (A=1,B=1)"
        }

        # CRITICAL: Use the correct order from specification
        # This order affects how patterns are numbered in the report
        for p in ["00", "10", "01", "11"]:  # Specific order matters!
            a = agg[p]
            total = a["TP"] + a["FN"] + a["TN"] + a["FP"]

            # Calculate performance metrics
            rows.append({
                "Pattern": pattern_names[p],
                "TP": int(a["TP"]),
                "FN": int(a["FN"]),
                "TN": int(a["TN"]),
                "FP": int(a["FP"]),
                "Accuracy": float((a["TP"] + a["TN"]) / total) if total else 0.0,
                "Sensitivity": float(a["TP"] / (a["TP"] + a["FN"])) if (a["TP"] + a["FN"]) else 0.0,
                "Specificity": float(a["TN"] / (a["TN"] + a["FP"])) if (a["TN"] + a["FP"]) else 0.0,
                "Events": int(total)
            })

        perf = pd.DataFrame(rows)

        # Add total row summarizing all patterns
        tot = perf[["TP", "FN", "TN", "FP"]].sum()
        den = int(tot.sum())

        total_row = pd.DataFrame([{
            "Pattern": "Total",
            "TP": int(tot["TP"]),
            "FN": int(tot["FN"]),
            "TN": int(tot["TN"]),
            "FP": int(tot["FP"]),
            "Accuracy": float((tot["TP"] + tot["TN"]) / den) if den else 0.0,
            "Sensitivity": float(tot["TP"] / (tot["TP"] + tot["FN"])) if (tot["TP"] + tot["FN"]) else 0.0,
            "Specificity": float(tot["TN"] / (tot["TN"] + tot["FP"])) if (tot["TN"] + tot["FP"]) else 0.0,
            "Events": den
        }])

        return pd.concat([perf, total_row], ignore_index=True)

    # Create XOR truth table for reference
    truth_df = pd.DataFrame(
        [(0,0,0),   # A=0, B=0 → E=0
         (1,0,1),   # A=1, B=0 → E=1
         (0,1,1),   # A=0, B=1 → E=1
         (1,1,0)],  # A=1, B=1 → E=0
        columns=["A","B","E"]
    )

    # Calculate performance for both GT and SUB
    gt_perf = _score_side(GT_TRIALS, GT_META, "GT")
    sub_perf = _score_side(SB_TRIALS, SB_META, "SUB")

    # Return with keys matching what the scoring system expects
    return {
        "truth_table": truth_df,
        "gt": gt_perf,
        "sub": sub_perf
    }

# =============================================================================
# Scoring System (Complete)
# =============================================================================
def compute_overall_score(metrics, config):
    """
    Compute weighted overall score from all metrics.

    Each metric is scored 0-10 based on quality thresholds, then
    weighted and combined into a final score out of 100.

    Args:
        metrics: Dictionary of computed metrics from all analyses
        config: Configuration with metric weights

    Returns:
        tuple: (overall_score, score_table, subscores_dict)
    """
    # Get metric weights from config
    weights = config.get("metric_weights", {})

    # Scoring functions for different metric types
    def _score_high(x, lo, hi):
        """
        Score metrics where higher is better.
        Maps [lo, hi] range to [0, 10] score.
        """
        if not np.isfinite(x):
            return 0.0
        return float(np.clip((x - lo) / max(1e-12, (hi - lo)), 0, 1) * 10.0)

    def _score_low(x, lo, hi):
        """
        Score metrics where lower is better.
        Maps [lo, hi] range to [10, 0] score.
        """
        if not np.isfinite(x):
            return 0.0
        return float((1.0 - np.clip((x - lo) / max(1e-12, (hi - lo)), 0, 1)) * 10.0)

    subscores = {}

    # Score 1: Raster Jaccard similarity
    if "raster" in metrics:
        mj = metrics["raster"]["mean_jaccard"]
        # Good range: 0.60-0.98, where 0.98+ is excellent
        subscores["raster_jaccard"] = _score_high(mj, lo=0.60, hi=0.98)

    # Score 2-3: PSTH correlation and RMSE
    if "psth" in metrics and "psth_metrics" in metrics["psth"]:
        psth_df = metrics["psth"]["psth_metrics"]

        # Extract correlation values
        r_vals = [x for x in psth_df["Overall_r"].values if np.isfinite(x)]
        # Good range: 0.75-0.98
        subscores["psth_corr"] = _score_high(np.mean(r_vals) if r_vals else 0,
                                            lo=0.75, hi=0.98)

        # Extract RMSE values
        rmse_vals = [x for x in psth_df["Overall_RMSE"].values if np.isfinite(x)]
        # Good range: 0.00-0.08 (lower is better)
        subscores["psth_rmse"] = _score_low(np.mean(rmse_vals) if rmse_vals else 0,
                                           lo=0.00, hi=0.08)

    # Score 4: KS statistic
    if "ks" in metrics and "ks_df" in metrics["ks"]:
        ks_vals = [x for x in metrics["ks"]["ks_df"]["KS_stat"].values if np.isfinite(x)]
        # Good range: 0.00-0.25 (lower is better)
        subscores["ks"] = _score_low(np.median(ks_vals) if ks_vals else 0,
                                    lo=0.00, hi=0.25)

    # Score 5: ISI CV difference
    if "isi" in metrics:
        gt_cv = np.array([x for x in metrics["isi"]["gt_cv"] if np.isfinite(x)])
        sub_cv = np.array([x for x in metrics["isi"]["sub_cv"] if np.isfinite(x)])

        if gt_cv.size > 0 and sub_cv.size > 0:
            # Calculate mean absolute difference in CV
            cv_delta = np.nanmean(np.abs(gt_cv - sub_cv))
            # Good range: 0.00-0.15 (lower is better)
            subscores["isi_cv_delta"] = _score_low(cv_delta, lo=0.00, hi=0.15)

    # Score 6: Fano factor difference
    if "fano" in metrics and "fano_df" in metrics["fano"]:
        fano_df = metrics["fano"]["fano_df"]
        gt_fano = [x for x in fano_df["ff_gt"].values if np.isfinite(x)]
        sub_fano = [x for x in fano_df["ff_sub"].values if np.isfinite(x)]

        if gt_fano and sub_fano:
            # Calculate mean absolute difference in Fano factors
            fano_delta = np.nanmean(np.abs(np.array(gt_fano) - np.array(sub_fano)))
            # Good range: 0.00-0.40 (lower is better)
            subscores["fano_delta"] = _score_low(fano_delta, lo=0.00, hi=0.40)

    # Score 7: Multi-scale correlation
    if "msc" in metrics and "msc_neuron_summary" in metrics["msc"]:
        if not metrics["msc"]["msc_neuron_summary"].empty:
            r_vals = [x for x in metrics["msc"]["msc_neuron_summary"]["r_mean"].values
                     if np.isfinite(x)]
            if r_vals:
                # Good range: 0.70-0.98
                subscores["multi_scale_corr"] = _score_high(np.mean(r_vals),
                                                           lo=0.70, hi=0.98)

    # Score 8: Schreiber similarity
    if "schreiber" in metrics and "schreiber_neuron_summary" in metrics["schreiber"]:
        if not metrics["schreiber"]["schreiber_neuron_summary"].empty:
            r_vals = [x for x in metrics["schreiber"]["schreiber_neuron_summary"]["r_mean"].values
                     if np.isfinite(x)]
            if r_vals:
                # Good range: 0.70-0.98
                subscores["schreiber"] = _score_high(np.mean(r_vals),
                                                    lo=0.70, hi=0.98)

    # Score 9: Granger causality Jaccard
    if "granger" in metrics and "granger_summary" in metrics["granger"]:
        if not metrics["granger"]["granger_summary"].empty:
            jacc_vals = [x for x in metrics["granger"]["granger_summary"]["jaccard"].values
                        if np.isfinite(x)]
            if jacc_vals:
                # Good range: 0.60-0.95
                subscores["granger_jaccard"] = _score_high(np.mean(jacc_vals),
                                                          lo=0.60, hi=0.95)

    # Score 10: Van Rossum distance
    if "vr" in metrics and "vr_neuron_summary" in metrics["vr"]:
        if not metrics["vr"]["vr_neuron_summary"].empty:
            vr_vals = [x for x in metrics["vr"]["vr_neuron_summary"]["VR_mean"].values
                      if np.isfinite(x)]
            if vr_vals:
                # Good range: 0.00-0.10 (lower is better)
                subscores["vr_distance"] = _score_low(np.mean(vr_vals),
                                                     lo=0.00, hi=0.10)

    # Score 11: Behavioral XOR accuracy
    if "behavior" in metrics:
        # Get total row from submission performance
        sub_total = metrics["behavior"]["sub"][metrics["behavior"]["sub"]["Pattern"] == "Total"]

        if not sub_total.empty:
            # Extract performance metrics
            acc = sub_total["Accuracy"].iloc[0]
            sens = sub_total["Sensitivity"].iloc[0]
            spec = sub_total["Specificity"].iloc[0]

            # Macro-average of the three metrics
            behavior_macro = np.mean([acc, sens, spec])
            # Good range: 0.70-0.98
            subscores["behavior"] = _score_high(behavior_macro, lo=0.70, hi=0.98)

    # Build score breakdown table
    rows = []
    weighted_sum = 0.0
    total_weight = 0.0

    for metric_name, weight in weights.items():
        if metric_name in subscores:
            score = subscores[metric_name]
            weighted = score * weight

            rows.append({
                "Metric": metric_name,
                "Subscore (0-10)": _fmt_decimal(score, 3),
                "Weight": weight,
                "Weighted Points": _fmt_decimal(weighted, 3)
            })

            weighted_sum += weighted
            total_weight += weight

    # Calculate overall score
    overall_10 = (weighted_sum / total_weight) if total_weight > 0 else 0.0
    overall_100 = _fmt_decimal(overall_10 * 10.0, 1)  # Scale to 0-100

    score_df = pd.DataFrame(rows)

    return overall_100, score_df, subscores

# =============================================================================
# Complete Main Report Builder with ALL Improvements
# =============================================================================
def build_report(GROUND_TRUTH, SUBMISSION, REPORT_PATH="report.pdf", CONFIG=None, META=None, verbose=True):
    """
    Build comprehensive PDF report comparing neural network simulations.

    This is the main entry point that orchestrates the entire analysis:
    1. Loads trial data from HDF5 files
    2. Preprocesses and aligns data
    3. Computes all metrics (14 different analyses)
    4. Calculates weighted scores
    5. Generates a detailed PDF report with visualizations

    Args:
        GROUND_TRUTH (str): Path to ground truth HDF5 file
        SUBMISSION (str): Path to submission HDF5 file
        REPORT_PATH (str): Output path for PDF report
        CONFIG (dict): Configuration parameters and weights
        META (dict): Metadata to include in report header
        verbose (bool): Whether to print progress messages

    Returns:
        dict: Summary containing report path, scores, metrics, and config
    """
    # Use provided config or create defaults with proper metric weights
    C = CONFIG or {}

    # Set default metric weights if not provided
    # These weights determine the relative importance of each metric
    if "metric_weights" not in C:
        C["metric_weights"] = {
            "raster_jaccard": 1.5,      # Spike timing overlap
            "psth_corr": 1.5,           # PSTH shape similarity
            "psth_rmse": 1.0,           # PSTH magnitude error
            "ks": 1.2,                  # Timing distribution
            "isi_cv_delta": 1.0,        # Firing regularity
            "fano_delta": 0.8,          # Count variability
            "multi_scale_corr": 1.0,    # Multi-scale similarity
            "schreiber": 1.0,           # Schreiber correlation
            "granger_jaccard": 0.8,     # Causal network similarity
            "vr_distance": 1.0,         # Van Rossum distance
            "behavior": 2.0             # XOR logic accuracy (highest weight)
        }

    META = META or {}

    # ===========================================================================
    # PHASE 1: Load and Preprocess Data
    # ===========================================================================

    _log("[load] Reading trials...", verbose)
    gt_trials = load_trials_new(GROUND_TRUTH)
    sub_trials = load_trials_new(SUBMISSION)

    _log("[preprocess] Extracting metadata and preparing data...", verbose)
    V = preprocess_trials(gt_trials, sub_trials, C)

    _log(f"[info] Loaded GT={len(gt_trials)} trials, SUB={len(sub_trials)} trials", verbose)
    _log(f"[info] Found {len(V['spike_cols'])} spike channels, {len(V['vm_cols'])} VM channels", verbose)

    # ===========================================================================
    # PHASE 2: Compute All Metrics
    # ===========================================================================

    metrics = {}

    # Metric 1: Raster/Jaccard Analysis
    if C.get("enable", {}).get("raster", True):
        _log("[metric] Computing raster/Jaccard...", verbose)
        metrics["raster"] = compute_raster_metrics(V, C)

    # Metric 2: PSTH Analysis
    if C.get("enable", {}).get("psth", True):
        _log("[metric] Computing PSTH...", verbose)
        metrics["psth"] = compute_psth_metrics(V, C)

    # Metric 3: KS Statistics
    if C.get("enable", {}).get("ks", True):
        _log("[metric] Computing KS statistics...", verbose)
        metrics["ks"] = compute_ks_metrics(V, C)

    # Metric 4: ISI Analysis
    if C.get("enable", {}).get("isi", True):
        _log("[metric] Computing ISI...", verbose)
        metrics["isi"] = compute_isi_metrics(V, C)

    # Metric 5: Fano Factor Analysis
    if C.get("enable", {}).get("fano", True):
        _log("[metric] Computing Fano factors...", verbose)
        metrics["fano"] = compute_fano_metrics(V, C)

    # Metric 6: VM Analysis
    if C.get("enable", {}).get("vm", True):
        _log("[metric] Computing VM analysis...", verbose)
        metrics["vm"] = compute_vm_metrics(V, C)

    # Metric 7: VM Zoom/Mismatch Analysis
    if C.get("enable", {}).get("vm_zoom", True):
        _log("[metric] Computing VM zoom/mismatch...", verbose)
        metrics["vm_zoom"] = compute_vm_zoom_metrics(V, C)

    # Metric 8: PSP Detection
    if C.get("enable", {}).get("psp", True):
        _log("[metric] Computing PSP detection...", verbose)
        metrics["psp"] = compute_psp_metrics(V, C)

    # Metric 9: Cross-Correlation
    if C.get("enable", {}).get("xcorr", True):
        _log("[metric] Computing cross-correlation...", verbose)
        metrics["xcorr"] = compute_xcorr_metrics(V, C)

    # Metric 10: Van Rossum Distance
    if C.get("enable", {}).get("vr", True):
        _log("[metric] Computing Van Rossum distances...", verbose)
        metrics["vr"] = compute_vr_metrics(V, C)

    # Metric 11: Multi-Scale Correlation
    if C.get("enable", {}).get("multi_scale_corr", True):
        _log("[metric] Computing multi-scale correlation...", verbose)
        metrics["msc"] = compute_msc_metrics(V, C)

    # Metric 12: Schreiber Similarity
    if C.get("enable", {}).get("schreiber", True):
        _log("[metric] Computing Schreiber similarity...", verbose)
        metrics["schreiber"] = compute_schreiber_metrics(V, C)

    # Metric 13: Granger Causality
    if C.get("enable", {}).get("granger", True):
        _log("[metric] Computing Granger causality...", verbose)
        metrics["granger"] = compute_granger_metrics(V, C)

    # Metric 14: Behavioral XOR Logic
    if C.get("enable", {}).get("behavior", True):
        _log("[metric] Computing behavioral XOR logic...", verbose)
        metrics["behavior"] = compute_behavioral_metrics(V, C)

    # ===========================================================================
    # PHASE 3: Compute Overall Scores
    # ===========================================================================

    _log("[scoring] Computing overall score...", verbose)
    overall_score, score_table, subscores = compute_overall_score(metrics, C)

    # ===========================================================================
    # PHASE 4: Build PDF Report
    # ===========================================================================

    _log("[pdf] Building report...", verbose)

    # Get ReportLab styles
    styles = getSampleStyleSheet()

    # Create PDF document with standard margins
    doc = SimpleDocTemplate(REPORT_PATH, pagesize=letter,
                           leftMargin=0.75*inch, rightMargin=0.75*inch,
                           topMargin=0.75*inch, bottomMargin=0.75*inch)

    # Flow list will contain all report elements
    flow = []

    # ---------------------------------------------------------------------------
    # Title Section
    # ---------------------------------------------------------------------------

    # Custom title style
    title_style = ParagraphStyle(
        'CustomTitle',
        parent=styles['Heading1'],
        fontSize=18,
        textColor=colors.HexColor('#1a1a1a'),
        spaceAfter=12,
        alignment=TA_CENTER
    )

    flow.append(Paragraph("WBE Challenge — XOR Network Comparison Report", title_style))
    flow.append(Spacer(1, 0.15 * inch))

    # Add metadata if provided
    if META:
        meta_line = " | ".join([f"{k}: {v}" for k, v in META.items()])
        flow.append(Paragraph(meta_line, styles["Normal"]))
        flow.append(Spacer(1, 0.05 * inch))

    # Add file information
    flow.append(Paragraph(f"<b>GT:</b> {Path(GROUND_TRUTH).name}", styles["Normal"]))
    flow.append(Paragraph(f"<b>SUB:</b> {Path(SUBMISSION).name}", styles["Normal"]))
    flow.append(Paragraph(f"<b>Trials:</b> GT={len(V['gt']['trials'])} | SUB={len(V['sub']['trials'])} | "
                         f"Neurons={len(V['spike_cols'])}", styles["Normal"]))
    flow.append(Spacer(1, 0.25 * inch))

    # ---------------------------------------------------------------------------
    # Overall Score Section
    # ---------------------------------------------------------------------------

    flow.append(Paragraph("<b>Overall Score</b>", styles["Heading1"]))
    flow.append(Paragraph(f"<b>Composite Score: {overall_score} / 100</b>", styles["Normal"]))
    flow.append(Spacer(1, 0.15 * inch))
    flow.append(Paragraph("<b>Score Breakdown</b>", styles["Heading2"]))
    flow.append(_table(score_table, compact_cols=True))
    flow.append(PageBreak())

    # ---------------------------------------------------------------------------
    # Section 1: RASTER ANALYSIS
    # ---------------------------------------------------------------------------

    if "raster" in metrics:
        flow.append(Paragraph("<b>1. Spike Raster Analysis</b>", styles["Heading1"]))
        flow.append(Spacer(1, 0.1 * inch))

        # Add insights
        flow.append(Paragraph("<b>Raster Insights</b>", styles["Heading2"]))
        flow.append(_bullets(metrics["raster"]["insights"], styles))
        flow.append(Spacer(1, 0.15 * inch))

        # Add raster plots
        flow.append(Image(metrics["raster"]["overlay_png"], width=7.5*inch, height=4.5*inch))
        flow.append(Spacer(1, 0.15 * inch))
        flow.append(Image(metrics["raster"]["diff_png"], width=7.5*inch, height=4.5*inch))
        flow.append(PageBreak())

    # ---------------------------------------------------------------------------
    # Section 2: PSTH ANALYSIS
    # ---------------------------------------------------------------------------

    if "psth" in metrics:
        flow.append(Paragraph("<b>2. PSTH Analysis</b>", styles["Heading1"]))
        flow.append(Spacer(1, 0.1 * inch))

        # Transposed metrics table
        flow.append(Paragraph("<b>PSTH Metrics</b>", styles["Heading2"]))
        flow.append(_table(metrics["psth"]["psth_metrics_transposed"], style='normal'))
        flow.append(Spacer(1, 0.15 * inch))

        # PSTH insights
        flow.append(Paragraph("<b>PSTH Insights</b>", styles["Heading2"]))
        flow.append(_bullets(metrics["psth"]["bullets"], styles))
        flow.append(Spacer(1, 0.15 * inch))

        # Add all PSTH plots
        flow.append(Paragraph("<b>PSTH Plots by Neuron and Pattern</b>", styles["Heading2"]))
        for i, plot in enumerate(metrics["psth"]["psth_plots"]):
            flow.append(Image(plot, width=7.5*inch, height=3.0*inch))
            flow.append(Spacer(1, 0.08 * inch))
            # Page break every 2 plots to avoid white space
            if (i + 1) % 2 == 0:
                flow.append(PageBreak())

        # Ensure page break after last plot if needed
        if len(metrics["psth"]["psth_plots"]) % 2 != 0:
            flow.append(PageBreak())

    # ---------------------------------------------------------------------------
    # Section 3: KS TEST
    # ---------------------------------------------------------------------------

    if "ks" in metrics:
        flow.append(Paragraph("<b>3. Kolmogorov-Smirnov Test</b>", styles["Heading1"]))
        flow.append(Spacer(1, 0.1 * inch))

        # Overall KS statistics
        flow.append(Paragraph("<b>KS Statistics (Overall)</b>", styles["Heading2"]))
        flow.append(_table(metrics["ks"]["ks_df"], compact_cols=False))
        flow.append(Spacer(1, 0.15 * inch))

        # KS issues
        flow.append(Paragraph("<b>KS Issues (Overall)</b>", styles["Heading2"]))
        flow.append(_bullets(metrics["ks"]["issue_bullets"], styles))
        flow.append(Spacer(1, 0.15 * inch))

        # Per-pattern flagged issues
        flow.append(Paragraph("<b>KS Flagged (Per-Pattern)</b>", styles["Heading2"]))
        flow.append(_bullets(metrics["ks"]["flagged_bullets"], styles))
        flow.append(PageBreak())

    # ---------------------------------------------------------------------------
    # Section 4: ISI ANALYSIS
    # ---------------------------------------------------------------------------

    if "isi" in metrics:
        flow.append(Paragraph("<b>4. Inter-Spike Interval Analysis</b>", styles["Heading1"]))
        flow.append(Spacer(1, 0.1 * inch))

        # ISI insights
        flow.append(Paragraph("<b>ISI Insights</b>", styles["Heading2"]))
        flow.append(_bullets(metrics["isi"]["insights"], styles))
        flow.append(Spacer(1, 0.15 * inch))

        # ISI plots
        flow.append(Image(metrics["isi"]["hist_png"], width=7.5*inch, height=6*inch))
        flow.append(Spacer(1, 0.15 * inch))
        flow.append(Image(metrics["isi"]["cv_png"], width=7.5*inch, height=4*inch))
        flow.append(PageBreak())

    # ---------------------------------------------------------------------------
    # Section 5: FANO FACTOR
    # ---------------------------------------------------------------------------

    if "fano" in metrics:
        flow.append(Paragraph("<b>5. Fano Factor Analysis</b>", styles["Heading1"]))
        flow.append(Spacer(1, 0.1 * inch))

        # Fano factor table
        flow.append(_table(metrics["fano"]["fano_df"], compact_cols=True))
        flow.append(Spacer(1, 0.15 * inch))

        # Fano issues if any
        if metrics["fano"]["insights"]:
            flow.append(Paragraph("<b>Fano Issues</b>", styles["Heading2"]))
            flow.append(_bullets(metrics["fano"]["insights"], styles))
            flow.append(Spacer(1, 0.15 * inch))

        # Fano plot
        flow.append(Image(metrics["fano"]["fano_png"], width=7*inch, height=4*inch))
        flow.append(PageBreak())

    # ---------------------------------------------------------------------------
    # Section 6: MEMBRANE POTENTIAL
    # ---------------------------------------------------------------------------

    if "vm" in metrics and "vm_plots" in metrics["vm"]:
        flow.append(Paragraph("<b>6. Membrane Potential Analysis</b>", styles["Heading1"]))
        flow.append(Spacer(1, 0.1 * inch))

        # Add all VM plots
        for i, plot in enumerate(metrics["vm"]["vm_plots"]):
            flow.append(Image(plot, width=7.5*inch, height=2.5*inch))
            if (i + 1) % 3 == 0:  # Page break every 3 plots
                flow.append(PageBreak())
            else:
                flow.append(Spacer(1, 0.08 * inch))

        # Ensure page break after last plot
        if len(metrics["vm"]["vm_plots"]) % 3 != 0:
            flow.append(PageBreak())

    # ---------------------------------------------------------------------------
    # Section 7: VM WORST MISMATCH
    # ---------------------------------------------------------------------------

    if "vm_zoom" in metrics:
        flow.append(Paragraph("<b>7. VM Worst Mismatch Analysis</b>", styles["Heading1"]))
        flow.append(Spacer(1, 0.1 * inch))

        # Worst per pattern
        flow.append(Paragraph("<b>Worst Mismatch per Pattern</b>", styles["Heading2"]))
        if not metrics["vm_zoom"]["pattern_worst_table"].empty:
            flow.append(_table(metrics["vm_zoom"]["pattern_worst_table"], compact_cols=True))
        else:
            flow.append(Paragraph("No mismatches found.", styles["Normal"]))
        flow.append(Spacer(1, 0.15 * inch))

        # Worst per neuron
        flow.append(Paragraph("<b>Worst Mismatch per Neuron</b>", styles["Heading2"]))
        if not metrics["vm_zoom"]["neuron_best_table"].empty:
            flow.append(_table(metrics["vm_zoom"]["neuron_best_table"], compact_cols=True))
        else:
            flow.append(Paragraph("No mismatches found.", styles["Normal"]))
        flow.append(Spacer(1, 0.15 * inch))

        # Add detailed trial comparison plots
        if "vm_zoom_plots" in metrics["vm_zoom"]:
            flow.append(Paragraph("<b>Detailed Trial Comparisons</b>", styles["Heading2"]))
            for plot in metrics["vm_zoom"]["vm_zoom_plots"]:
                flow.append(Image(plot, width=7.5*inch, height=5*inch))
                flow.append(Spacer(1, 0.1 * inch))

        flow.append(PageBreak())

    # ---------------------------------------------------------------------------
    # Section 8: PSP DETECTION
    # ---------------------------------------------------------------------------

    if "psp" in metrics:
        flow.append(Paragraph("<b>8. PSP Event Detection</b>", styles["Heading1"]))
        flow.append(Spacer(1, 0.1 * inch))

        # Add table for each pattern
        for pattern, table in metrics["psp"]["pattern_tables"].items():
            flow.append(Paragraph(f"<b>Pattern {pattern}</b>", styles["Heading2"]))
            flow.append(_table(table, compact_cols=True))
            flow.append(Spacer(1, 0.1 * inch))
        flow.append(PageBreak())

    # ---------------------------------------------------------------------------
    # Section 9: CROSS-CORRELATION
    # ---------------------------------------------------------------------------

    if "xcorr" in metrics:
        flow.append(Paragraph("<b>9. Cross-Correlation Analysis</b>", styles["Heading1"]))
        flow.append(Spacer(1, 0.1 * inch))

        # Zero-lag summary
        flow.append(Paragraph("<b>Zero-Lag Summary</b>", styles["Heading2"]))
        flow.append(_table(metrics["xcorr"]["ccg_summary"], compact_cols=True))
        flow.append(Spacer(1, 0.15 * inch))

        # Add all CCG matrix plots
        flow.append(Paragraph("<b>Cross-Correlation Matrices</b>", styles["Heading2"]))
        for plot in metrics["xcorr"]["ccg_plots"]:
            flow.append(Image(plot, width=7*inch, height=7*inch))
            flow.append(PageBreak())

    # ---------------------------------------------------------------------------
    # Section 10: VAN ROSSUM
    # ---------------------------------------------------------------------------

    if "vr" in metrics:
        flow.append(Paragraph("<b>10. Van Rossum Distance</b>", styles["Heading1"]))
        flow.append(Spacer(1, 0.1 * inch))

        # Pattern summary
        flow.append(Paragraph("<b>Pattern Summary</b>", styles["Heading2"]))
        if not metrics["vr"]["vr_pattern_summary"].empty:
            flow.append(_table(metrics["vr"]["vr_pattern_summary"], compact_cols=True))
        else:
            flow.append(Paragraph("No data available.", styles["Normal"]))
        flow.append(Spacer(1, 0.15 * inch))

        # Neuron summary
        flow.append(Paragraph("<b>Neuron Summary</b>", styles["Heading2"]))
        if not metrics["vr"]["vr_neuron_summary"].empty:
            flow.append(_table(metrics["vr"]["vr_neuron_summary"], compact_cols=True))
        else:
            flow.append(Paragraph("No data available.", styles["Normal"]))
        flow.append(PageBreak())

    # ---------------------------------------------------------------------------
    # Section 11: MULTI-SCALE CORRELATION
    # ---------------------------------------------------------------------------

    if "msc" in metrics:
        flow.append(Paragraph("<b>11. Multi-Scale Correlation</b>", styles["Heading1"]))
        flow.append(Spacer(1, 0.1 * inch))

        # Pattern summary
        if "msc_pattern_summary" in metrics["msc"] and not metrics["msc"]["msc_pattern_summary"].empty:
            flow.append(Paragraph("<b>Pattern Summary</b>", styles["Heading2"]))
            flow.append(_table(metrics["msc"]["msc_pattern_summary"], compact_cols=True))
            flow.append(Spacer(1, 0.15 * inch))

        # Neuron summary
        if "msc_neuron_summary" in metrics["msc"] and not metrics["msc"]["msc_neuron_summary"].empty:
            flow.append(Paragraph("<b>Neuron Summary</b>", styles["Heading2"]))
            flow.append(_table(metrics["msc"]["msc_neuron_summary"], compact_cols=True))
            flow.append(Spacer(1, 0.15 * inch))

        # Add all MSC plots
        if "msc_plots" in metrics["msc"]:
            flow.append(Paragraph("<b>Correlation vs Smoothing Scale</b>", styles["Heading2"]))
            for plot in metrics["msc"]["msc_plots"]:
                flow.append(Image(plot, width=7*inch, height=4.5*inch))
                flow.append(Spacer(1, 0.1 * inch))

        flow.append(PageBreak())

    # ---------------------------------------------------------------------------
    # Section 12: SCHREIBER SIMILARITY
    # ---------------------------------------------------------------------------

    if "schreiber" in metrics:
        flow.append(Paragraph("<b>12. Schreiber Similarity</b>", styles["Heading1"]))
        flow.append(Spacer(1, 0.1 * inch))

        # Pattern summary
        if "schreiber_pattern_summary" in metrics["schreiber"] and not metrics["schreiber"]["schreiber_pattern_summary"].empty:
            flow.append(Paragraph("<b>Pattern Summary</b>", styles["Heading2"]))
            flow.append(_table(metrics["schreiber"]["schreiber_pattern_summary"], compact_cols=True))
            flow.append(Spacer(1, 0.15 * inch))

        # Neuron summary
        if "schreiber_neuron_summary" in metrics["schreiber"] and not metrics["schreiber"]["schreiber_neuron_summary"].empty:
            flow.append(Paragraph("<b>Neuron Summary</b>", styles["Heading2"]))
            flow.append(_table(metrics["schreiber"]["schreiber_neuron_summary"], compact_cols=True))
            flow.append(Spacer(1, 0.15 * inch))

        # Add all Schreiber plots
        if "schreiber_plots" in metrics["schreiber"]:
            flow.append(Paragraph("<b>Similarity by Pattern</b>", styles["Heading2"]))
            for plot in metrics["schreiber"]["schreiber_plots"]:
                flow.append(Image(plot, width=7*inch, height=4*inch))
                flow.append(Spacer(1, 0.1 * inch))

        flow.append(PageBreak())

    # ---------------------------------------------------------------------------
    # Section 13: GRANGER CAUSALITY
    # ---------------------------------------------------------------------------

    if "granger" in metrics:
        flow.append(Paragraph("<b>13. Granger Causality</b>", styles["Heading1"]))
        flow.append(Spacer(1, 0.1 * inch))

        # Pattern summary
        if "granger_summary" in metrics["granger"] and not metrics["granger"]["granger_summary"].empty:
            flow.append(Paragraph("<b>Pattern Summary</b>", styles["Heading2"]))
            flow.append(_table(metrics["granger"]["granger_summary"], compact_cols=True))
            flow.append(Spacer(1, 0.15 * inch))

        # Add all Granger heatmaps
        if "granger_plots" in metrics["granger"]:
            flow.append(Paragraph("<b>Causal Network Heatmaps</b>", styles["Heading2"]))
            for plot in metrics["granger"]["granger_plots"]:
                flow.append(Image(plot, width=7.5*inch, height=4*inch))
                flow.append(PageBreak())

    # ---------------------------------------------------------------------------
    # Section 14: BEHAVIORAL XOR LOGIC
    # ---------------------------------------------------------------------------

    if "behavior" in metrics:
        flow.append(Paragraph("<b>14. XOR Behavioral Logic</b>", styles["Heading1"]))
        flow.append(Spacer(1, 0.1 * inch))

        # Truth table
        flow.append(Paragraph("<b>Truth Table</b>", styles["Heading2"]))
        flow.append(_table(metrics["behavior"]["truth_table"], compact_cols=True))
        flow.append(Spacer(1, 0.2 * inch))

        # Ground truth performance
        flow.append(Paragraph("<b>Ground Truth Performance</b>", styles["Heading2"]))
        flow.append(_table(metrics["behavior"]["gt"], compact_cols=False))
        flow.append(Spacer(1, 0.2 * inch))

        # Submission performance
        flow.append(Paragraph("<b>Submission Performance</b>", styles["Heading2"]))
        flow.append(_table(metrics["behavior"]["sub"], compact_cols=False))

    # ===========================================================================
    # PHASE 5: Generate PDF
    # ===========================================================================

    try:
        # Build the PDF document
        doc.build(flow)
        _log(f"[done] Report saved to {REPORT_PATH}", verbose)
    except Exception as e:
        _log(f"[error] Failed to build PDF: {e}", verbose)
        raise

    # Return summary information
    return {
        "report_path": REPORT_PATH,
        "overall_score": overall_score,
        "score_table": score_table,
        "metrics": metrics,
        "config": C,
        "meta": META
    }

[setup] installing scikit-learn …
[setup] install complete.

[setup] package versions:
  - numpy         2.0.2
  - pandas        2.2.2
  - matplotlib    3.10.0
  - seaborn       0.13.2
  - h5py          3.15.1
  - scipy         1.16.3
  - scikit-learn  1.6.1
  - reportlab     4.4.5



#  Run Code

In [2]:
# === RUN CODE for WBE XOR Report (Jainil Shah) ===
"""
Analyzes and compares two neural network XOR simulations.
Generates PDF report with 14 metrics for spike timing, membrane potentials,
network dynamics, and behavioral accuracy.
"""

# File paths
GROUND_TRUTH = "/content/drive/MyDrive/InDomainData/DataGeneration/Data/InDomainXOR_balanced_400trials_100ms.hdf5"  # Reference data
SUBMISSION = "/content/drive/MyDrive/InDomainData/DataGeneration/Data/InDomainXOR_SUB_balanced_400trials_100ms.hdf5"  # Test data
REPORT_PATH = "/content/report.pdf"  # Output PDF

# Configuration
CONFIG = {
    # Trial processing
    "trial_length": 100,          # Trial duration (ms)
    "stim_window": (5, 60),       # Expected input pulse window (ms)

    # Spike metrics
    "bin_size": 5,                # PSTH bin size (ms)
    "psth_align_pre": 10,         # Pre-stimulus baseline (ms)
    "psth_align_post": 80,        # Post-stimulus window (ms)

    # Kolmogorov-Smirnov test
    "ks_window_mode": "full",     # "aligned" | "full" | "guarded_full"
    "ks_alpha": 0.05,             # Significance level
    "ks_min_pooled_spikes": 40,   # Minimum spikes for reliable test

    # Inter-spike intervals
    "isi_keep_cross_trial": True, # Include cross-trial ISIs
    "isi_max_keep_ms": 600,       # Maximum ISI to keep (ms)
    "isi_hist_bins": 48,          # Histogram bins

    # Fano factor
    "fano_patterns": ["00", "01", "10", "11"],  # XOR stimulus patterns
    "fano_default_window": (5, 95),             # Default counting window

    # Membrane potential processing
    "kernel_size": 101,           # Smoothing kernel (odd number)
    "vm_filter_kernel": 101,      # Median filter size
    "vm_use_filtered": True,      # Use filtered VM for analysis

    # PSP detection
    "psp_peak_prominence": 0.5,   # Minimum peak height (mV)
    "psp_min_peak_distance": 2,   # Minimum time between peaks (ms)
    "psp_baseline_pre": 10,       # Baseline period (ms)
    "psp_clip_to_window": True,   # Only count PSPs in response window

    # VM mismatch analysis
    "vm_zoom_window_ms": 3000,    # Window size for mismatch plots (ms)
    "vm_zoom_topk": 1,            # Number of worst mismatches to show
    "vm_zoom_min_gap_ms": 1500,   # Minimum gap between windows (ms)

    # Cross-correlation
    "xcorr_max_lag": 15,          # Maximum lag (ms)

    # Van Rossum distance
    "vr_tau_ms": 20.0,            # Time constant (ms)
    "vr_min_trials": 1,           # Minimum trials needed

    # Multi-scale correlation
    "msc_sigma_range": (1, 101),  # Smoothing scale range (ms)
    "msc_per_pattern": True,      # Analyze each pattern separately

    # Schreiber similarity
    "schreiber_sigma_ms": 10.0,   # Fixed smoothing scale (ms)
    "schreiber_min_trials": 1,    # Minimum trials needed

    # Granger causality
    "granger_bin_ms": 5,          # Time bin size (ms)
    "granger_lag_bins": 10,       # AR model lag (bins)
    "granger_alpha": 0.05,        # Significance level
    "granger_min_bins": 50,       # Minimum bins for analysis

    # XOR behavioral logic
    "behavior_resp_lo": 3,        # Response window start (ms)
    "behavior_resp_hi": 40,       # Response window end (ms)
    "behavior_stim_eps": 1e-12,   # Stimulus detection threshold

    # Metric weights (higher = more important)
    "metric_weights": {
        "raster_jaccard": 8,       # Spike-time overlap
        "psth_corr": 12,          # Firing rate profile similarity
        "psth_rmse": 4,           # Firing rate magnitude error
        "ks": 6,                  # Spike distribution changes
        "isi_cv_delta": 3,        # Firing regularity changes
        "fano_delta": 3,          # Trial-to-trial variability
        "vm_rms": 6,              # Membrane potential error
        "psp_count_delta": 2,     # Synaptic event differences
        "xcorr_zero_lag": 2,      # Synchronization changes
        "vr_distance": 2,         # Combined timing/count metric
        "multi_scale_corr": 8,    # Multi-scale similarity
        "schreiber": 8,           # Smoothed correlation
        "granger_jaccard": 2,     # Causal network similarity
        "behavior": 12,           # XOR accuracy (highest weight)
    },

    # Enable/disable sections (set False to skip)
    "enable": {
        "summary": True,           # Overall summary
        "raster": True,            # Spike raster plots
        "psth": True,              # PSTH analysis
        "psth_aligned": True,      # Response-aligned PSTH
        "ks": True,                # Kolmogorov-Smirnov test
        "isi": True,               # Inter-spike intervals
        "fano": True,              # Fano factor
        "vm": True,                # Membrane potentials
        "vm_zoom": True,           # VM mismatch analysis
        "psp": True,               # PSP detection
        "xcorr": True,             # Cross-correlation
        "vr": True,                # Van Rossum distance
        "multi_scale_corr": True,  # Multi-scale correlation
        "schreiber": True,         # Schreiber similarity
        "granger": True,           # Granger causality
        "behavior": True,          # XOR logic
    },
}

# Metadata
META = {
    "id": "WBE-2025-XOR",           # Analysis ID
    "team": "YourTeam",             # Team name
    "submission": "xor-test-001",   # Submission version
    "date": "2025-01-01"           # Analysis date
}

# Run analysis
out = build_report(
    GROUND_TRUTH,    # Reference data
    SUBMISSION,      # Test data
    REPORT_PATH,     # Output PDF
    CONFIG=CONFIG,   # Configuration
    META=META,       # Metadata
    verbose=True     # Show progress
)

print("Saved:", out["report_path"])

# Additional outputs available:
# out["overall_score"]  - Final score (0-100)
# out["score_table"]    - Individual metric scores
# out["metrics"]        - Raw metric values
# out["config"]         - Configuration used
# out["meta"]           - Metadata included

[load] Reading trials...
[preprocess] Extracting metadata and preparing data...
[info] Loaded GT=400 trials, SUB=400 trials
[info] Found 5 spike channels, 5 VM channels
[metric] Computing raster/Jaccard...
[metric] Computing PSTH...
[metric] Computing KS statistics...
[metric] Computing ISI...
[metric] Computing Fano factors...
[metric] Computing VM analysis...
[metric] Computing VM zoom/mismatch...
[metric] Computing PSP detection...
[metric] Computing cross-correlation...
[metric] Computing Van Rossum distances...
[metric] Computing multi-scale correlation...
[metric] Computing Schreiber similarity...
[metric] Computing Granger causality...
[metric] Computing behavioral XOR logic...
[scoring] Computing overall score...
[pdf] Building report...
[done] Report saved to /content/report.pdf
Saved: /content/report.pdf
